In [ ]:
# | default_exp preprocessing.ocr

In [ ]:
%load_ext autoreload
%autoreload 2

# Document and image OCR preprocessing

> Recursively recognize PDF pages and PNG/JPEG images with local Ollama or Alibaba Cloud DashScope.

Each `.pdf`, `.png`, `.jpg`, or `.jpeg` source produces one UTF-8 Markdown document under a `.md` directory at the source root. Relative directories are preserved, PDF pages may run concurrently while their Markdown sections remain ordered, and multiple source files run concurrently through bounded asynchronous clients. Hidden directories and `*.assets` bundles are excluded so generated crops and alternate OCR output trees are never rediscovered as source images. Ollama remains the default provider; DashScope uses its OpenAI-compatible multimodal interface when selected.

`layout_mode="plain"` keeps the original one-request-per-page transcription path. The opt-in `layout_mode="pp-doclayout"` path uses the official GLM-OCR PP-DocLayout-V3 detector, recognizes text/table/formula regions, and preserves figure/chart pixels without making figure-model requests. It renders every region in detector reading order with page, label, bounding-box, and status markers; tables and figures retain linked source crops. The schema-v2 `<document>.layout.json` sidecar remains the machine-readable layout representation. Install its optional runtime once with `uv sync --extra ocr-layout`; the detector model is downloaded on first use.

Structured runs write `<document>.partial.md` and `<document>.partial.layout.json` checkpoints as soon as layout is detected and refresh them after every region. Compatible interrupted work resumes automatically. Text-region requests are capped at 1,024 tokens, while `LAYOUT_MAX_OUTPUT_TOKENS` remains the upper bound for tables and formulas. When GLM-OCR returns a correct prefix and then degenerates into repeated lines, character blocks, Markdown fences, or duplicate tables, the first stable text unit or complete HTML table is retained with region status `recovered`; its crop and recovery reason remain in the sidecar without adding a visible warning. Only genuinely incomplete or unrecoverable output becomes a failed region and makes the document `partial`. A folder-wide advisory lock prevents two updated notebook kernels from competing for the same output tree. Existing legacy `.layout-*` directories are deliberately left untouched because an older kernel may still own them.

## Concurrency tuning

### Local Ollama interface (measured)

Measured on 2026-07-22 with local `glm-ocr:latest` on an RTX 4090 with 24 GiB VRAM, 123 GiB system RAM, and 32 logical CPUs. The 1.1B F16 model was loaded entirely on the GPU with a 4.8 GB runner footprint and a 32,768-token effective context. Ollama is configured with `OLLAMA_NUM_PARALLEL=4`, Flash Attention, and a `q4_0` KV cache.

A steady-state benchmark submitted eight distinct 200-DPI document pages through the same async image-chat interface used below:

| Request concurrency | Throughput | Mean latency |
| ---: | ---: | ---: |
| 1 | 0.924 pages/s | 1.08 s |
| 2 | 1.998 pages/s | 0.95 s |
| 4 | 3.508 pages/s | 0.94 s |
| 6 | 3.537 pages/s | 1.27 s |

Four concurrent requests are the throughput/latency optimum for the current local interface. Raising the client limit to six produced almost no additional throughput because the Ollama runner processes at most four requests; the extra requests wait in its queue and increase latency. Ollama also documents that parallel requests multiply context-memory requirements ([concurrency documentation](https://docs.ollama.com/faq#how-does-ollama-handle-concurrent-requests)).

The batch configuration uses both `http://127.0.0.1:11434` and `http://172.27.74.16:11434`. A preflight checks that each endpoint serves the selected model, then an available-slot pool contributes four request slots per healthy host. With both endpoints healthy, `MAX_CONCURRENCY = 8`; if one is unavailable, it is reported and the pool safely falls back to four slots instead of overloading the remaining host. Faster endpoints naturally receive more work because their slots return to the pool sooner. Output and checkpoint ordering remain deterministic regardless of which host handles a request. Four slots are measured for the local endpoint and used as a conservative starting assumption for the second endpoint; reduce `OLLAMA_MAX_CONCURRENCY` if monitoring shows that its runner has a lower parallel-request limit.

### DashScope OpenAI-compatible interface (quota-based)

The current `.env` selects the China (Beijing) DashScope domain, `https://dashscope.aliyuncs.com/compatible-mode/v1`, and resolves to `qwen3.7-plus` because `OPENAILIKED_OCR_MODEL` is unset. Alibaba documents this domain as the Beijing DashScope endpoint; workspace-dedicated domains are recommended for production workloads that need higher, more stable concurrency ([regional domains](https://www.alibabacloud.com/help/en/model-studio/regions/)). No billable DashScope benchmark was run for this notebook, so the following values are conservative starting points rather than measured optima.

For `qwen3.7-plus` in the Chinese mainland, the published account-level limits are 30,000 requests/minute and 5,000,000 input-plus-output tokens/minute. Usage is aggregated across the root account, including its API keys, RAM users, and workspaces. The service may also enforce per-second RPS/TPS limits and dynamic traffic-burst protection, so a sudden batch can receive HTTP 429 even while below the minute totals ([rate limits](https://www.alibabacloud.com/help/en/model-studio/rate-limit), [rate-limit best practices](https://www.alibabacloud.com/help/en/model-studio/rate-limiting-best-practices)).

Each page is sent as one Base64 data URL. DashScope limits the encoded image to 10 MB, which this notebook enforces with PNG-to-JPEG fallback. Qwen3.7 accepts up to 16,777,216 pixels, but its default non-high-resolution policy uses `max_pixels=2,621,440`; larger pages may be downscaled. A typical 200-DPI A4 page is about 3.9 million pixels, so fine-print OCR may benefit from `vl_high_resolution_images=true` or a higher `max_pixels`, at the cost of more visual tokens, latency, and billed usage. Qwen3.7 uses approximately one visual token per `32 x 32` pixels ([visual-input limits](https://www.alibabacloud.com/help/en/model-studio/vision)). Plain mode retains the default policy; PP-DocLayout mode sends its smaller region crops with high-resolution vision enabled.

Start DashScope folder runs at `MAX_CONCURRENCY = 8` and `PAGE_CONCURRENCY = 2`. If monitoring shows no 429 responses, timeouts, or rising p95 latency, test 12 and then 16 global requests; retain an increase only when throughput improves materially. Reduce to 4/1 or 4/2 when burst errors occur. For one large PDF, 8/4 is a conservative starting point. The current implementation isolates failed files but does not automatically retry 429 responses; rerunning with `OVERWRITE = False` resumes missing outputs. Alibaba also supports the `X-DashScope-Wait-Timeout` header for burst queuing, but this notebook does not currently enable it.

### Recommended settings by interface

| Interface and workload | `MAX_CONCURRENCY` | `PAGE_CONCURRENCY` | Basis |
| --- | ---: | ---: | --- |
| Ollama, two-host mixed recursive folder | 8 | 2 | Four configured request slots per healthy host |
| Ollama, two hosts and one large PDF | 8 | 4 | Four pages can issue region requests across both hosts |
| Ollama, single-host mixed recursive folder | 4 | 2 | Measured optimum on this RTX 4090 host |
| Ollama, single host and one large PDF | 4 | 4 | Lets one document occupy all four runner slots |
| Ollama, single host with fairness priority | 4 | 1 | One request per active document |
| DashScope, mixed recursive folder | 8 | 2 | Conservative unbilled starting point |
| DashScope, one large PDF | 8 | 4 | Conservative unbilled starting point |

In [ ]:
# | export
import asyncio
import base64
import copy
import fcntl
import json
import os
import shutil
import warnings
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime
from html import escape
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory
from time import perf_counter
from typing import Any, Awaitable, Callable, Iterator, Literal, Sequence, cast

import pymupdf
from dotenv import load_dotenv
from ollama import AsyncClient as AsyncOllamaClient
from ollama import Client as OllamaClient
from openai import AsyncOpenAI, OpenAI
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# | export
def _find_project_root() -> Path:
    """Find the nearest parent containing pyproject.toml."""
    starts: list[Path] = []
    module_file = globals().get("__file__")
    if isinstance(module_file, str):
        starts.append(Path(module_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "pyproject.toml").is_file():
                return candidate
    return Path.cwd().resolve()


PROJ_ROOT = _find_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)

In [ ]:
# | export
OCRProvider = Literal["ollama", "dashscope"]
OCRLayoutMode = Literal["plain", "pp-doclayout"]
_OCRClient = OllamaClient | OpenAI


def _normalize_ollama_hosts(host: str | Sequence[str]) -> tuple[str, ...]:
    """Return unique, non-empty Ollama endpoint URLs in configured order."""
    candidates = (host,) if isinstance(host, str) else tuple(host)
    if not candidates:
        raise ValueError("At least one Ollama host is required")

    normalized: list[str] = []
    seen: set[str] = set()
    for candidate in candidates:
        if not isinstance(candidate, str) or not candidate.strip():
            raise ValueError("Ollama hosts must be non-empty strings")
        endpoint = candidate.strip().rstrip("/")
        if endpoint not in seen:
            normalized.append(endpoint)
            seen.add(endpoint)
    return tuple(normalized)


@dataclass(frozen=True)
class _OllamaEndpoint:
    host: str
    client: Any


class _AsyncOllamaPool:
    """Route requests through the next available slot across healthy hosts."""

    def __init__(
        self,
        hosts: str | Sequence[str],
        *,
        request_timeout_s: float,
        max_concurrency: int,
        client_factory: Callable[[str, float], Any] | None = None,
    ) -> None:
        self.configured_hosts = _normalize_ollama_hosts(hosts)
        if max_concurrency <= 0:
            raise ValueError("max_concurrency must be positive")
        self.slots_per_host = max(
            1,
            (max_concurrency + len(self.configured_hosts) - 1)
            // len(self.configured_hosts),
        )
        factory = client_factory or (
            lambda endpoint, timeout: AsyncOllamaClient(
                host=endpoint, timeout=timeout
            )
        )
        self._endpoints = tuple(
            _OllamaEndpoint(host, factory(host, request_timeout_s))
            for host in self.configured_hosts
        )
        self._slots: asyncio.Queue[_OllamaEndpoint] = asyncio.Queue()
        self.active_hosts: tuple[str, ...] = ()
        self.unavailable_hosts: tuple[tuple[str, str], ...] = ()

    @property
    def request_slots(self) -> int:
        """Number of concurrent requests supplied by currently healthy hosts."""
        return len(self.active_hosts) * self.slots_per_host

    async def show(self, model: str) -> Any:
        """Preflight every host, retaining any endpoint that serves the model."""
        checks = await asyncio.gather(
            *(endpoint.client.show(model) for endpoint in self._endpoints),
            return_exceptions=True,
        )
        healthy: list[tuple[_OllamaEndpoint, Any]] = []
        unavailable: list[tuple[str, str]] = []
        for endpoint, check in zip(self._endpoints, checks, strict=True):
            if isinstance(check, BaseException):
                unavailable.append((endpoint.host, str(check)))
            else:
                healthy.append((endpoint, check))
        if not healthy:
            details = "; ".join(
                f"{host}: {error}" for host, error in unavailable
            )
            raise RuntimeError(f"No configured Ollama host is available ({details})")

        self._slots = asyncio.Queue()
        for _slot in range(self.slots_per_host):
            for endpoint, _ in healthy:
                self._slots.put_nowait(endpoint)
        self.active_hosts = tuple(endpoint.host for endpoint, _ in healthy)
        self.unavailable_hosts = tuple(unavailable)
        return healthy[0][1]

    async def chat(self, *args: Any, **kwargs: Any) -> Any:
        """Run one chat request on the next endpoint with an available slot."""
        if not self.active_hosts:
            raise RuntimeError("Ollama host pool has not been preflighted")
        endpoint = await self._slots.get()
        try:
            return await endpoint.client.chat(*args, **kwargs)
        finally:
            self._slots.put_nowait(endpoint)

    async def close(self) -> None:
        """Close all endpoint clients owned by the pool."""
        await asyncio.gather(
            *(endpoint.client.close() for endpoint in self._endpoints),
            return_exceptions=True,
        )


_AsyncOCRClient = AsyncOllamaClient | AsyncOpenAI | _AsyncOllamaPool


@dataclass(frozen=True)
class LayoutRegion:
    """One PP-DocLayout-V3 region in rendered-page pixel coordinates."""

    index: int
    label: str
    score: float
    bbox: tuple[int, int, int, int]
    task_type: Literal["text", "table", "formula", "figure"]


@dataclass(frozen=True)
class OCRResult:
    """Outcome of attempting to convert one source file to Markdown."""

    pdf_path: Path
    markdown_path: Path
    status: Literal["processed", "partial", "skipped", "failed"]
    pages_total: int = 0
    pages_completed: int = 0
    error: str | None = None
    regions_total: int = 0
    regions_completed: int = 0
    regions_failed: int = 0

    @property
    def source_path(self) -> Path:
        """Source PDF or image path (preferred provider-neutral name)."""
        return self.pdf_path


@dataclass(frozen=True)
class _OCRMarkdown:
    markdown: str
    recovery: str | None = None


In [ ]:
# | export
def _resolve_root(root_folder: Path | str) -> Path:
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"OCR root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"OCR root is not a directory: {root}")
    return root


_SUPPORTED_SOURCE_SUFFIXES = frozenset({".pdf", ".png", ".jpg", ".jpeg"})
_IMAGE_SUFFIXES = frozenset({".png", ".jpg", ".jpeg"})


def _is_generated_ocr_path(path: Path, root: Path) -> bool:
    """Exclude hidden output/work trees and published OCR asset bundles."""
    directory_parts = path.relative_to(root).parts[:-1]
    return any(
        part.startswith(".") or part.casefold().endswith(".assets")
        for part in directory_parts
    )


def _ocr_jobs(root: Path) -> list[tuple[Path, Path]]:
    """Return deterministic source/target pairs and reject target collisions."""
    output_root = root / ".md"
    source_files = [
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.casefold() in _SUPPORTED_SOURCE_SUFFIXES
        and not _is_generated_ocr_path(path, root)
    ]
    source_files.sort(
        key=lambda path: (
            path.relative_to(root).as_posix().casefold(),
            path.relative_to(root).as_posix(),
        )
    )

    jobs: list[tuple[Path, Path]] = []
    targets: dict[str, Path] = {}
    for source_path in source_files:
        relative_path = source_path.relative_to(root).with_suffix(".md")
        markdown_path = output_root / relative_path
        collision_key = markdown_path.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"OCR output collision: {previous} and {source_path} both map to {markdown_path}"
            )
        targets[collision_key] = source_path
        jobs.append((source_path, markdown_path))
    return jobs


In [ ]:
# | export
_DEFAULT_OLLAMA_MODEL = "glm-ocr"
_DEFAULT_DASHSCOPE_MODEL = "qwen3.7-plus"
_DEFAULT_DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
_DEFAULT_LAYOUT_MODEL = "PaddlePaddle/PP-DocLayoutV3_safetensors"
_DEFAULT_LAYOUT_MAX_OUTPUT_TOKENS = 4096
_DEFAULT_LAYOUT_TEXT_MAX_OUTPUT_TOKENS = 1024
_LAYOUT_RESPONSE_VALIDATION_VERSION = 2
_OLLAMA_PROMPT = "Text Recognition:"
_OLLAMA_LAYOUT_PROMPTS = {
    "text": "Text Recognition:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
}
_DASHSCOPE_PROMPT = (
    "Convert this document page to Markdown. Preserve the reading order, headings, "
    "paragraphs, lists, tables, code, and formulas. Return only the Markdown "
    "transcription without commentary."
)
_DASHSCOPE_LAYOUT_PROMPT = "qwenvl markdown"
_DASHSCOPE_MAX_DATA_URL_BYTES = 10 * 1024 * 1024
_LAYOUT_LABEL_TASKS = {
    "text": [
        "abstract",
        "algorithm",
        "aside_text",
        "content",
        "doc_title",
        "figure_title",
        "footer",
        "footnote",
        "formula_number",
        "header",
        "number",
        "paragraph_title",
        "reference",
        "reference_content",
        "seal",
        "text",
        "vertical_text",
        "vision_footnote",
    ],
    "table": ["table"],
    "formula": ["display_formula", "inline_formula"],
    # The official GLM-OCR pipeline preserves figure pixels without asking the
    # text model to describe them.  Treating figures as OCR requests can make
    # GLM-OCR generate unbounded, repetitive Markdown fences.
    "skip": ["chart", "footer_image", "header_image", "image"],
}
_LAYOUT_ASSET_TASKS = frozenset({"table", "formula", "figure"})


def _resolve_provider(provider: OCRProvider | str) -> OCRProvider:
    normalized = provider.strip().casefold()
    if normalized not in {"ollama", "dashscope"}:
        raise ValueError("provider must be 'ollama' or 'dashscope'")
    return cast(OCRProvider, normalized)


def _resolve_layout_mode(layout_mode: OCRLayoutMode | str) -> OCRLayoutMode:
    normalized = layout_mode.strip().casefold()
    if normalized not in {"plain", "pp-doclayout"}:
        raise ValueError("layout_mode must be 'plain' or 'pp-doclayout'")
    return cast(OCRLayoutMode, normalized)


def _resolve_model(provider: OCRProvider, model: str | None) -> str:
    if model is not None:
        if not model.strip():
            raise ValueError("model must not be empty")
        return model.strip()
    if provider == "dashscope":
        return (
            os.getenv("OPENAILIKED_OCR_MODEL", "").strip() or _DEFAULT_DASHSCOPE_MODEL
        )
    return _DEFAULT_OLLAMA_MODEL


def _resolve_prompt(provider: OCRProvider, prompt: str | None) -> str:
    if prompt is not None:
        if not prompt.strip():
            raise ValueError("prompt must not be empty")
        return prompt.strip()
    return _DASHSCOPE_PROMPT if provider == "dashscope" else _OLLAMA_PROMPT


def _resolve_layout_prompt(
    provider: OCRProvider,
    task_type: str,
    prompt: str | None,
) -> str:
    if prompt is not None:
        return _resolve_prompt(provider, prompt)
    if provider == "dashscope":
        return _DASHSCOPE_LAYOUT_PROMPT
    return _OLLAMA_LAYOUT_PROMPTS.get(task_type, _OLLAMA_PROMPT)


def create_pp_doclayout_detector(
    *,
    model_name: str = _DEFAULT_LAYOUT_MODEL,
    threshold: float = 0.3,
    device: str | None = None,
) -> Any:
    """Load and start the official GLM-OCR PP-DocLayout-V3 detector.

    Install the optional runtime with ``uv sync --extra ocr-layout``. The
    model is downloaded on first use unless ``model_name`` names a local path.
    """
    if not model_name.strip():
        raise ValueError("layout_model must not be empty")
    if not 0 <= threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    try:
        from glmocr.config import LayoutConfig
        from glmocr.layout import PPDocLayoutDetector
    except ImportError as error:
        raise RuntimeError(
            "PP-DocLayout-V3 requires the optional OCR layout runtime; "
            "run `uv sync --extra ocr-layout`"
        ) from error

    config = LayoutConfig(
        model_dir=model_name.strip(),
        threshold=threshold,
        batch_size=1,
        device=device,
        label_task_mapping=_LAYOUT_LABEL_TASKS,
    )
    detector = PPDocLayoutDetector(config)
    detector.start()
    return detector


def _response_content(response: object, provider: OCRProvider) -> str:
    if provider == "ollama":
        message = getattr(response, "message", None)
    else:
        choices = getattr(response, "choices", None)
        message = getattr(choices[0], "message", None) if choices else None
    content = getattr(message, "content", None)
    if not isinstance(content, str) or not content.strip():
        raise ValueError(f"{provider} OCR returned an empty response")
    return content.strip()


def _response_finish_reason(response: object, provider: OCRProvider) -> str | None:
    if provider == "ollama":
        finish_reason = getattr(response, "done_reason", None)
    else:
        choices = getattr(response, "choices", None)
        finish_reason = getattr(choices[0], "finish_reason", None) if choices else None
    return finish_reason if isinstance(finish_reason, str) else None


def _layout_repetition_reason(content: str) -> str | None:
    fence_count = content.count("```")
    if fence_count >= 12:
        return f"contains {fence_count} Markdown fences"

    lines = [line.strip() for line in content.splitlines() if line.strip()]
    if lines:
        repeated_line, line_count = Counter(lines).most_common(1)[0]
        if line_count >= 8:
            return (
                f"repeats one {len(repeated_line)}-character line "
                f"{line_count} times"
            )

    compact = "".join(content.split())
    block_size = 32
    if len(compact) >= block_size * 2:
        blocks = Counter(
            compact[index : index + block_size]
            for index in range(len(compact) - block_size + 1)
        )
        _, block_count = blocks.most_common(1)[0]
        if block_count >= 8:
            return (
                f"repeats a {block_size}-character block "
                f"{block_count} times"
            )
    return None


def _semantic_line_matches(left: str, right: str) -> bool:
    if left == right:
        return True
    shorter, longer = sorted((left, right), key=len)
    return (
        len(shorter) >= 4
        and len(shorter) / len(longer) >= 0.8
        and shorter in longer
    )


def _cut_at_repeated_character_block(content: str) -> str:
    block_size = 32
    compact_characters: list[str] = []
    source_indices: list[int] = []
    for source_index, character in enumerate(content):
        if not character.isspace():
            compact_characters.append(character)
            source_indices.append(source_index)
    compact = "".join(compact_characters)
    if len(compact) < block_size * 2:
        return content
    blocks = Counter(
        compact[index : index + block_size]
        for index in range(len(compact) - block_size + 1)
    )
    repeated_block, block_count = blocks.most_common(1)[0]
    if block_count < 8:
        return content
    first = compact.find(repeated_block)
    second = compact.find(repeated_block, first + 1)
    if second <= 0:
        return content
    return content[: source_indices[second]].rstrip()


def _recover_repeated_text(
    content: str,
    *,
    single_line: bool = False,
) -> str | None:
    semantic_lines = [
        line.strip()
        for line in content.splitlines()
        if line.strip()
        and not line.strip().startswith(("```", "~~~"))
    ]
    normalized_lines = [
        "".join(
            character.casefold()
            for character in line
            if character.isalnum()
        )
        for line in semantic_lines
    ]
    line_counts = Counter(normalized_lines)
    selected_lines: list[str] = []
    selected_normalized: list[str] = []
    for line, normalized in zip(semantic_lines, normalized_lines, strict=True):
        matching_index = next(
            (
                index
                for index, previous in enumerate(selected_normalized)
                if _semantic_line_matches(normalized, previous)
            ),
            None,
        )
        if matching_index is not None:
            if len(normalized) > len(selected_normalized[matching_index]):
                selected_lines[matching_index] = line
                selected_normalized[matching_index] = normalized
            break
        if line_counts[normalized] >= 8:
            if not selected_lines:
                selected_lines.append(line)
            break
        selected_lines.append(line)
        selected_normalized.append(normalized)

    if single_line:
        selected_lines = selected_lines[:1]
    candidate = _cut_at_repeated_character_block(
        "\n".join(selected_lines).strip()
    )
    if single_line:
        candidate = candidate.lstrip("•●·- \t")
    if (
        not candidate
        or not any(character.isalnum() for character in candidate)
        or _layout_repetition_reason(candidate) is not None
    ):
        return None
    return candidate


def _recover_complete_html_table(content: str) -> str | None:
    casefolded = content.casefold()
    start = casefolded.find("<table")
    if start < 0:
        return None
    end = casefolded.find("</table>", start)
    if end < 0:
        return None
    return content[start : end + len("</table>")].strip() or None


def _validated_layout_content(
    response: object,
    provider: OCRProvider,
    content: str,
    task_type: str,
    label: str | None = None,
) -> tuple[str, str | None]:
    finish_reason = _response_finish_reason(response, provider)
    repetition_reason = (
        _layout_repetition_reason(content) if task_type == "text" else None
    )
    recovered: str | None = None
    if repetition_reason is not None:
        recovered = _recover_repeated_text(
            content,
            single_line=label
            in {
                "doc_title",
                "figure_title",
                "footer",
                "formula_number",
                "header",
                "number",
                "paragraph_title",
            },
        )
    elif task_type == "table" and finish_reason == "length":
        recovered = _recover_complete_html_table(content)
    if recovered is not None:
        reason = repetition_reason or "output continued after a complete HTML table"
        return recovered, (
            f"Recovered the first complete non-repeating {task_type} unit after "
            f"{provider} degeneration: {reason}"
        )
    if (
        repetition_reason is not None
        and task_type == "text"
        and label in {"footer", "header", "number"}
    ):
        return "", (
            f"Discarded a degenerate {provider} response for a decorative "
            f"or empty {label} region: {repetition_reason}"
        )
    if repetition_reason is not None:
        raise ValueError(
            f"{provider} OCR returned a pathological repeated response: "
            f"{repetition_reason}"
        )
    if finish_reason == "length":
        raise ValueError(
            f"{provider} OCR reached its output-token limit before completion"
        )
    return content, None


def _layout_output_token_limit(task_type: str, maximum: int) -> int:
    if task_type == "text":
        return min(maximum, _DEFAULT_LAYOUT_TEXT_MAX_OUTPUT_TOKENS)
    return maximum


def _data_url(image_bytes: bytes, mime_type: str) -> str:
    encoded = base64.b64encode(image_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def _dashscope_image_data_url(pixmap: pymupdf.Pixmap) -> str:
    png_data_url = _data_url(pixmap.tobytes("png"), "image/png")
    if len(png_data_url.encode("ascii")) <= _DASHSCOPE_MAX_DATA_URL_BYTES:
        return png_data_url

    jpeg_data_url = _data_url(
        pixmap.tobytes("jpeg", jpg_quality=90),
        "image/jpeg",
    )
    if len(jpeg_data_url.encode("ascii")) <= _DASHSCOPE_MAX_DATA_URL_BYTES:
        return jpeg_data_url
    raise ValueError(
        "Image payload exceeds DashScope's 10 MiB Base64 input limit; "
        "reduce PDF dpi or the source image dimensions"
    )


def _pixmap_markdown(
    pixmap: pymupdf.Pixmap,
    provenance: str,
    *,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    high_resolution: bool = False,
    max_output_tokens: int | None = None,
    layout_task_type: str | None = None,
    layout_label: str | None = None,
) -> _OCRMarkdown:
    if provider == "ollama":
        options: dict[str, int | float] = {"temperature": 0}
        if max_output_tokens is not None:
            options.update(
                {"num_predict": max_output_tokens, "repeat_penalty": 1.1}
            )
        response = client.chat(  # type: ignore[operator]
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                    "images": [pixmap.tobytes("png")],
                }
            ],
            options=options,
        )
    else:
        extra_body: dict[str, bool] = {"enable_thinking": False}
        if high_resolution:
            extra_body["vl_high_resolution_images"] = True
        arguments: dict[str, Any] = {
            "model": model,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {"url": _dashscope_image_data_url(pixmap)},
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            "temperature": 0,
            "extra_body": extra_body,
        }
        if max_output_tokens is not None:
            arguments["max_tokens"] = max_output_tokens
        response = client.chat.completions.create(  # type: ignore[union-attr]
            **arguments,
        )
    content = _response_content(response, provider)
    recovery: str | None = None
    if layout_task_type is not None:
        content, recovery = _validated_layout_content(
            response,
            provider,
            content,
            layout_task_type,
            layout_label,
        )
    return _OCRMarkdown(
        f"<!-- {provenance} -->\n\n{content}",
        recovery,
    )


async def _pixmap_markdown_async(
    pixmap: pymupdf.Pixmap,
    provenance: str,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    request_semaphore: asyncio.Semaphore | None = None,
    high_resolution: bool = False,
    max_output_tokens: int | None = None,
    layout_task_type: str | None = None,
    layout_label: str | None = None,
) -> _OCRMarkdown:
    async def send_request() -> object:
        if provider == "ollama":
            options: dict[str, int | float] = {"temperature": 0}
            if max_output_tokens is not None:
                options.update(
                    {"num_predict": max_output_tokens, "repeat_penalty": 1.1}
                )
            return await client.chat(  # type: ignore[union-attr]
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                        "images": [pixmap.tobytes("png")],
                    }
                ],
                options=options,
            )
        extra_body: dict[str, bool] = {"enable_thinking": False}
        if high_resolution:
            extra_body["vl_high_resolution_images"] = True
        arguments: dict[str, Any] = {
            "model": model,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {"url": _dashscope_image_data_url(pixmap)},
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            "temperature": 0,
            "extra_body": extra_body,
        }
        if max_output_tokens is not None:
            arguments["max_tokens"] = max_output_tokens
        return await client.chat.completions.create(  # type: ignore[union-attr]
            **arguments,
        )

    if request_semaphore is None:
        response = await send_request()
    else:
        async with request_semaphore:
            response = await send_request()
    content = _response_content(response, provider)
    recovery: str | None = None
    if layout_task_type is not None:
        content, recovery = _validated_layout_content(
            response,
            provider,
            content,
            layout_task_type,
            layout_label,
        )
    return _OCRMarkdown(
        f"<!-- {provenance} -->\n\n{content}",
        recovery,
    )


def _page_markdown(
    page: pymupdf.Page,
    page_number: int,
    *,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    dpi: int,
) -> str:
    return _pixmap_markdown(
        page.get_pixmap(dpi=dpi, alpha=False),
        f"Page {page_number}",
        client=client,
        provider=provider,
        model=model,
        prompt=prompt,
    ).markdown


async def _page_markdown_async(
    page: pymupdf.Page,
    page_number: int,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str,
    dpi: int,
    request_semaphore: asyncio.Semaphore | None = None,
) -> str:
    result = await _pixmap_markdown_async(
        page.get_pixmap(dpi=dpi, alpha=False),
        f"Page {page_number}",
        client=client,
        provider=provider,
        model=model,
        prompt=prompt,
        request_semaphore=request_semaphore,
    )
    return result.markdown


def _load_image_pixmap_with_pymupdf(path: Path) -> pymupdf.Pixmap:
    return pymupdf.Pixmap(str(path))


def _load_image_pixmap_with_pillow(path: Path) -> pymupdf.Pixmap:
    """Decode an image with Pillow and composite transparency onto white."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with Image.open(path) as image:
                image.load()
                rgba = image.convert("RGBA")
                background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
                rgb = Image.alpha_composite(background, rgba).convert("RGB")
    except (Image.DecompressionBombWarning, Image.DecompressionBombError) as error:
        raise ValueError(
            f"Image exceeds Pillow's {Image.MAX_IMAGE_PIXELS:,}-pixel safety "
            f"limit: {path}"
        ) from error
    return pymupdf.Pixmap(
        pymupdf.csRGB, rgb.width, rgb.height, rgb.tobytes(), False
    )


def _load_image_pixmap(path: Path) -> pymupdf.Pixmap:
    try:
        pixmap = _load_image_pixmap_with_pymupdf(path)
    except Exception as pymupdf_error:
        try:
            return _load_image_pixmap_with_pillow(path)
        except Exception as pillow_error:
            raise ValueError(
                f"Could not decode image {path}: PyMuPDF: {pymupdf_error}; "
                f"Pillow: {pillow_error}"
            ) from pillow_error

    if pixmap.colorspace not in {pymupdf.csGRAY, pymupdf.csRGB}:
        pixmap = pymupdf.Pixmap(pymupdf.csRGB, pixmap)
    if pixmap.alpha:
        try:
            return _load_image_pixmap_with_pillow(path)
        except Exception:
            pixmap = pymupdf.Pixmap(pixmap, 0)
    return pixmap


def _pixmap_to_pil(pixmap: pymupdf.Pixmap) -> Image.Image:
    mode = "RGBA" if pixmap.alpha else ("L" if pixmap.n == 1 else "RGB")
    return Image.frombytes(mode, (pixmap.width, pixmap.height), pixmap.samples).convert(
        "RGB"
    )


def _pil_to_pixmap(image: Image.Image) -> pymupdf.Pixmap:
    rgb = image.convert("RGB")
    return pymupdf.Pixmap(pymupdf.csRGB, rgb.width, rgb.height, rgb.tobytes(), False)


def _crop_pil_image(
    image: Image.Image,
    bbox: tuple[int, int, int, int],
) -> Image.Image:
    """Crop an already-decoded page without Pillow's decompression warning."""
    left, top, right, bottom = bbox
    if not (
        0 <= left < right <= image.width
        and 0 <= top < bottom <= image.height
    ):
        raise ValueError(
            f"Layout crop {bbox} is outside the {image.width}x{image.height} page"
        )
    # Pillow applies its file-decompression warning to Image.crop as well.
    # This image is already decoded and the validated crop cannot exceed it.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", Image.DecompressionBombWarning)
        return image.crop(bbox)


def _layout_task_type(label: str, declared: object) -> str:
    if declared in {"skip", "figure"}:
        return "figure"
    if isinstance(declared, str) and declared in _LAYOUT_LABEL_TASKS:
        return cast(str, declared)
    for task_type, labels in _LAYOUT_LABEL_TASKS.items():
        if label in labels:
            return "figure" if task_type == "skip" else task_type
    return "text"


def _detect_layout_regions(
    detector: Any,
    image: Image.Image,
) -> list[LayoutRegion]:
    """Run a started GLM-OCR PPDocLayoutDetector for one rendered page."""
    if not hasattr(detector, "process"):
        raise TypeError("layout_detector must provide process(images, ...)")
    output = detector.process([image], save_visualization=False)
    pages = output[0] if isinstance(output, tuple) else output
    if not isinstance(pages, list) or len(pages) != 1:
        raise ValueError("PP-DocLayout returned an unexpected page result")
    raw_regions = pages[0]
    if not isinstance(raw_regions, list):
        raise ValueError("PP-DocLayout returned malformed regions")

    sortable: list[tuple[int, int, dict[str, Any]]] = []
    for position, raw_region in enumerate(raw_regions):
        if not isinstance(raw_region, dict):
            raise ValueError("PP-DocLayout returned a malformed region")
        declared_order = raw_region.get("index", position)
        order = declared_order if isinstance(declared_order, int) else position
        sortable.append((order, position, raw_region))
    sortable.sort(key=lambda item: (item[0], item[1]))

    regions: list[LayoutRegion] = []
    for reading_order, (_, _, raw_region) in enumerate(sortable, start=1):
        label = str(raw_region.get("label", "text")).strip() or "text"
        score = float(raw_region.get("score", 0.0))
        if "bbox_2d" in raw_region:
            raw_bbox = raw_region["bbox_2d"]
            if not isinstance(raw_bbox, (list, tuple)) or len(raw_bbox) != 4:
                raise ValueError("PP-DocLayout returned an invalid bbox_2d")
            x1, y1, x2, y2 = (
                float(raw_bbox[0]) * image.width / 1000,
                float(raw_bbox[1]) * image.height / 1000,
                float(raw_bbox[2]) * image.width / 1000,
                float(raw_bbox[3]) * image.height / 1000,
            )
        else:
            raw_bbox = raw_region.get("coordinate", raw_region.get("bbox"))
            if not isinstance(raw_bbox, (list, tuple)) or len(raw_bbox) != 4:
                raise ValueError(
                    "PP-DocLayout returned a region without a bounding box"
                )
            x1, y1, x2, y2 = (float(value) for value in raw_bbox)
        bbox = (
            max(0, min(image.width, int(round(x1)))),
            max(0, min(image.height, int(round(y1)))),
            max(0, min(image.width, int(round(x2)))),
            max(0, min(image.height, int(round(y2)))),
        )
        if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
            continue
        task_type = _layout_task_type(label, raw_region.get("task_type"))
        regions.append(
            LayoutRegion(
                reading_order,
                label,
                score,
                bbox,
                cast(Literal["text", "table", "formula", "figure"], task_type),
            )
        )
    if not regions:
        regions.append(
            LayoutRegion(1, "text", 0.0, (0, 0, image.width, image.height), "text")
        )
    return regions


def _layout_asset_path(
    target: Path,
    page_number: int,
    region: LayoutRegion | None,
) -> Path:
    if region is None:
        filename = f"page-{page_number:04d}.png"
    else:
        label = (
            "".join(
                character if character.isalnum() else "-" for character in region.label
            ).strip("-")
            or "region"
        )
        filename = f"page-{page_number:04d}-region-{region.index:03d}-{label}.png"
    return Path(f"{target.stem}.assets") / filename


def _recognized_content(section: str) -> str:
    _, separator, content = section.partition("\n\n")
    return content.strip() if separator else section.strip()


def _render_layout_region(
    page_number: int,
    record: dict[str, Any],
    *,
    asset_prefix: str = "",
) -> str:
    index = int(record["index"])
    label = str(record["label"])
    task_type = str(record["task_type"])
    status = str(record.get("status", "pending"))
    bbox = [int(value) for value in record["bbox"]]
    bbox_text = ",".join(str(value) for value in bbox)
    provenance = (
        f"layout-region page={page_number} index={index} label={label} "
        f"bbox={bbox_text} status={status}"
    )
    blocks: list[str] = []
    asset = record.get("asset")
    asset_link = f"{asset_prefix}{asset}" if isinstance(asset, str) else None
    if asset_link and task_type in {"figure", "table"}:
        kind = "Figure" if task_type == "figure" else "Table"
        alt = label.replace("_", " ").strip().title() or kind
        blocks.append(f"![{alt} {index}](<{asset_link}>)")
    elif asset_link and status == "failed":
        blocks.append(f"![Failed {label} region {index}](<{asset_link}>)")

    if status == "pending":
        blocks.append(f"> [!NOTE] OCR pending for `{label}` region {index}.")
    elif status == "failed":
        error = str(record.get("error") or "unknown OCR failure").replace("\n", " ")
        blocks.append(
            f"> [!WARNING] OCR failed for `{label}` region {index}: {error}"
        )

    normalized = str(record.get("content") or "").strip()
    if normalized:
        if label == "doc_title" and not normalized.startswith("#"):
            normalized = f"# {normalized}"
        elif label == "paragraph_title" and not normalized.startswith("#"):
            normalized = f"## {normalized}"
        elif label == "list" and not normalized.startswith(("- ", "* ", "1. ")):
            normalized = "\n".join(
                f"- {line.strip()}"
                for line in normalized.splitlines()
                if line.strip()
            )
        elif task_type == "formula" and not normalized.startswith("$$"):
            normalized = f"$$\n{normalized}\n$$"
        blocks.append(normalized)

    return f"<!-- {provenance} -->\n\n" + (
        "\n\n".join(blocks) if blocks else f"<!-- Empty {label} region -->"
    )


def _render_layout_page(
    page_record: dict[str, Any] | None,
    page_number: int,
    *,
    asset_prefix: str = "",
) -> str:
    if page_record is None:
        return f"<!-- Page {page_number}: layout pending -->"
    sections: list[str] = []
    page_asset = page_record.get("page_asset")
    if isinstance(page_asset, str):
        sections.append(
            f"![Original page {page_number}](<{asset_prefix}{page_asset}>)"
        )
    sections.extend(
        _render_layout_region(page_number, record, asset_prefix=asset_prefix)
        for record in page_record.get("regions", [])
    )
    return f"<!-- Page {page_number} -->\n\n" + "\n\n".join(sections)


def _layout_counts(layout: dict[str, Any]) -> tuple[int, int, int]:
    regions = [
        region
        for page in layout.get("pages", [])
        if isinstance(page, dict)
        for region in page.get("regions", [])
    ]
    completed = sum(
        region.get("status") in {"completed", "preserved", "recovered"}
        for region in regions
    )
    failed = sum(region.get("status") == "failed" for region in regions)
    return len(regions), completed, failed


def _render_layout_document(
    layout: dict[str, Any],
    *,
    partial: bool,
    asset_prefix: str = "",
) -> str:
    pages = layout.get("pages", [])
    sections: list[str] = []
    if partial:
        pages_completed = sum(
            isinstance(page, dict) and page.get("status") in {"completed", "partial"}
            for page in pages
        )
        regions_total, regions_completed, regions_failed = _layout_counts(layout)
        sections.append(
            "> [!NOTE] OCR processing is incomplete. "
            f"Pages: {pages_completed}/{layout['pages_total']}; "
            f"regions: {regions_completed}/{regions_total} complete, "
            f"{regions_failed} failed."
        )
    sections.extend(
        _render_layout_page(page, page_number, asset_prefix=asset_prefix)
        for page_number, page in enumerate(pages, start=1)
    )
    return "\n\n".join(sections).rstrip() + "\n"


def _layout_region_record(
    region: LayoutRegion,
    page_number: int,
    target: Path,
) -> dict[str, Any]:
    asset_path = (
        _layout_asset_path(target, page_number, region)
        if region.task_type in _LAYOUT_ASSET_TASKS
        else None
    )
    return {
        "index": region.index,
        "label": region.label,
        "score": region.score,
        "bbox": list(region.bbox),
        "task_type": region.task_type,
        "status": "preserved" if region.task_type == "figure" else "pending",
        "prompt": None,
        "asset": asset_path.as_posix() if asset_path is not None else None,
        "content": "",
        "error": None,
        "recovery": None,
    }


def _regions_from_page_record(page_record: dict[str, Any]) -> list[LayoutRegion]:
    return [
        LayoutRegion(
            int(record["index"]),
            str(record["label"]),
            float(record.get("score", 0.0)),
            cast(tuple[int, int, int, int], tuple(int(v) for v in record["bbox"])),
            cast(
                Literal["text", "table", "formula", "figure"],
                str(record["task_type"]),
            ),
        )
        for record in page_record.get("regions", [])
    ]


def _save_region_asset(
    crop: pymupdf.Pixmap,
    record: dict[str, Any],
    *,
    target: Path,
    page_number: int,
    region: LayoutRegion,
    stage_assets: Path,
) -> None:
    if record.get("asset") is None:
        record["asset"] = _layout_asset_path(
            target, page_number, region
        ).as_posix()
    asset_file = stage_assets.parent / str(record["asset"])
    asset_file.parent.mkdir(parents=True, exist_ok=True)
    if not asset_file.is_file():
        asset_file.write_bytes(crop.tobytes("png"))


def _prepare_layout_page(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    target: Path,
    stage_assets: Path,
    detector: Any,
    provider: OCRProvider,
    prompt: str | None,
    embed_page_image: bool,
    max_output_tokens: int,
    existing_record: dict[str, Any] | None,
    detected_regions: list[LayoutRegion] | None = None,
) -> tuple[Image.Image, list[LayoutRegion], dict[str, Any]]:
    image = _pixmap_to_pil(pixmap)
    if existing_record is None:
        regions = detected_regions or _detect_layout_regions(detector, image)
        page_record: dict[str, Any] = {
            "page_number": page_number,
            "width": image.width,
            "height": image.height,
            "status": "processing",
            "error": None,
            "regions": [
                _layout_region_record(region, page_number, target) for region in regions
            ],
        }
    else:
        page_record = copy.deepcopy(existing_record)
        page_record["status"] = "processing"
        regions = _regions_from_page_record(page_record)

    if embed_page_image:
        page_asset = _layout_asset_path(target, page_number, None)
        page_file = stage_assets.parent / page_asset
        page_file.parent.mkdir(parents=True, exist_ok=True)
        if not page_file.is_file():
            page_file.write_bytes(pixmap.tobytes("png"))
        page_record["page_asset"] = page_asset.as_posix()

    records_by_index = {
        int(record["index"]): record for record in page_record["regions"]
    }
    for region in regions:
        record = records_by_index[region.index]
        if region.task_type != "figure":
            record["prompt"] = _resolve_layout_prompt(
                provider, region.task_type, prompt
            )
            record["max_output_tokens"] = _layout_output_token_limit(
                region.task_type, max_output_tokens
            )
        if region.task_type in _LAYOUT_ASSET_TASKS:
            _save_region_asset(
                _pil_to_pixmap(_crop_pil_image(image, region.bbox)),
                record,
                target=target,
                page_number=page_number,
                region=region,
                stage_assets=stage_assets,
            )
    return image, regions, page_record


def _layout_page_sync(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    target: Path,
    stage_assets: Path,
    detector: Any,
    client: _OCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str | None,
    embed_page_image: bool,
    max_output_tokens: int,
    existing_record: dict[str, Any] | None = None,
    checkpoint: Callable[[dict[str, Any]], None] | None = None,
) -> tuple[str, dict[str, Any]]:
    image, regions, page_record = _prepare_layout_page(
        pixmap,
        page_number,
        target=target,
        stage_assets=stage_assets,
        detector=detector,
        provider=provider,
        prompt=prompt,
        embed_page_image=embed_page_image,
        max_output_tokens=max_output_tokens,
        existing_record=existing_record,
    )
    if checkpoint is not None:
        checkpoint(page_record)
    records_by_index = {
        int(record["index"]): record for record in page_record["regions"]
    }
    for region in regions:
        record = records_by_index[region.index]
        if record["status"] in {"completed", "preserved", "recovered"}:
            continue
        crop = _pil_to_pixmap(_crop_pil_image(image, region.bbox))
        try:
            response = _pixmap_markdown(
                crop,
                f"Page {page_number} / Region {region.index}",
                client=client,
                provider=provider,
                model=model,
                prompt=str(record["prompt"]),
                high_resolution=True,
                max_output_tokens=int(record["max_output_tokens"]),
                layout_task_type=region.task_type,
                layout_label=region.label,
            )
            if response.recovery is not None:
                _save_region_asset(
                    crop,
                    record,
                    target=target,
                    page_number=page_number,
                    region=region,
                    stage_assets=stage_assets,
                )
            record.update(
                status=(
                    "recovered"
                    if response.recovery is not None
                    else "completed"
                ),
                content=_recognized_content(response.markdown),
                error=None,
                recovery=response.recovery,
            )
        except Exception as error:
            _save_region_asset(
                crop,
                record,
                target=target,
                page_number=page_number,
                region=region,
                stage_assets=stage_assets,
            )
            record.update(
                status="failed",
                content="",
                error=f"{type(error).__name__}: {error}",
            )
        if checkpoint is not None:
            checkpoint(page_record)
    page_record["status"] = (
        "partial"
        if any(record["status"] == "failed" for record in page_record["regions"])
        else "completed"
    )
    if checkpoint is not None:
        checkpoint(page_record)
    return _render_layout_page(page_record, page_number), page_record


async def _layout_page_async(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    target: Path,
    stage_assets: Path,
    detector: Any,
    client: _AsyncOCRClient,
    provider: OCRProvider,
    model: str,
    prompt: str | None,
    embed_page_image: bool,
    request_semaphore: asyncio.Semaphore | None,
    layout_semaphore: asyncio.Semaphore | None,
    max_output_tokens: int,
    existing_record: dict[str, Any] | None = None,
    checkpoint: Callable[[dict[str, Any]], Awaitable[None]] | None = None,
) -> tuple[str, dict[str, Any]]:
    image = _pixmap_to_pil(pixmap)
    if existing_record is not None:
        regions = _regions_from_page_record(existing_record)
    elif layout_semaphore is None:
        regions = await asyncio.to_thread(_detect_layout_regions, detector, image)
    else:
        async with layout_semaphore:
            regions = await asyncio.to_thread(_detect_layout_regions, detector, image)
    image, regions, page_record = _prepare_layout_page(
        pixmap,
        page_number,
        target=target,
        stage_assets=stage_assets,
        detector=detector,
        provider=provider,
        prompt=prompt,
        embed_page_image=embed_page_image,
        max_output_tokens=max_output_tokens,
        existing_record=existing_record,
        detected_regions=regions,
    )
    if checkpoint is not None:
        await checkpoint(page_record)
    records_by_index = {
        int(record["index"]): record for record in page_record["regions"]
    }

    async def recognize(
        region: LayoutRegion,
    ) -> tuple[int, str, str | None, str | None]:
        record = records_by_index[region.index]
        crop = _pil_to_pixmap(_crop_pil_image(image, region.bbox))
        try:
            response = await _pixmap_markdown_async(
                crop,
                f"Page {page_number} / Region {region.index}",
                client=client,
                provider=provider,
                model=model,
                prompt=str(record["prompt"]),
                request_semaphore=request_semaphore,
                high_resolution=True,
                max_output_tokens=int(record["max_output_tokens"]),
                layout_task_type=region.task_type,
                layout_label=region.label,
            )
            if response.recovery is not None:
                _save_region_asset(
                    crop,
                    record,
                    target=target,
                    page_number=page_number,
                    region=region,
                    stage_assets=stage_assets,
                )
            return (
                region.index,
                "recovered" if response.recovery is not None else "completed",
                _recognized_content(response.markdown),
                response.recovery,
            )
        except Exception as error:
            _save_region_asset(
                crop,
                record,
                target=target,
                page_number=page_number,
                region=region,
                stage_assets=stage_assets,
            )
            return (
                region.index,
                "failed",
                f"{type(error).__name__}: {error}",
                None,
            )

    tasks = [
        asyncio.create_task(recognize(region))
        for region in regions
        if records_by_index[region.index]["status"]
        not in {"completed", "preserved", "recovered"}
    ]
    try:
        for completed_task in asyncio.as_completed(tasks):
            region_index, status, value, recovery = await completed_task
            record = records_by_index[region_index]
            if status in {"completed", "recovered"}:
                record.update(
                    status=status,
                    content=value or "",
                    error=None,
                    recovery=recovery,
                )
            else:
                record.update(
                    status=status,
                    content="",
                    error=value,
                    recovery=None,
                )
            if checkpoint is not None:
                await checkpoint(page_record)
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
    page_record["status"] = (
        "partial"
        if any(record["status"] == "failed" for record in page_record["regions"])
        else "completed"
    )
    if checkpoint is not None:
        await checkpoint(page_record)
    return _render_layout_page(page_record, page_number), page_record


def _publish_layout_bundle(
    stage_root: Path,
    target: Path,
    markdown: str,
    layout: dict[str, Any],
) -> None:
    """Publish assets and sidecar first, then atomically expose Markdown last."""
    stage_markdown = stage_root / target.name
    stage_sidecar = stage_root / f"{target.stem}.layout.json"
    stage_assets = stage_root / f"{target.stem}.assets"
    stage_assets.mkdir(parents=True, exist_ok=True)
    stage_markdown.write_text(markdown, encoding="utf-8", newline="\n")
    stage_sidecar.write_text(
        json.dumps(layout, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )

    final_sidecar = target.with_suffix(".layout.json")
    final_assets = target.parent / stage_assets.name
    publications = [
        (stage_assets, final_assets),
        (stage_sidecar, final_sidecar),
        (stage_markdown, target),
    ]
    backups: list[tuple[Path, Path]] = []
    published: list[tuple[Path, Path]] = []
    try:
        for backup_index, (_, final_path) in enumerate(publications):
            if final_path.exists():
                backup_path = stage_root / f"backup-{backup_index}"
                final_path.replace(backup_path)
                backups.append((backup_path, final_path))
        for staged_path, final_path in publications:
            staged_path.replace(final_path)
            published.append((final_path, staged_path))
    except Exception:
        for final_path, staged_path in reversed(published):
            if final_path.exists():
                final_path.replace(staged_path)
        for backup_path, final_path in reversed(backups):
            if backup_path.exists():
                backup_path.replace(final_path)
        raise


def _layout_workspace_paths(target: Path) -> tuple[Path, Path, Path, Path, Path]:
    work_root = target.parent / f".{target.stem}.layout-work"
    assets = work_root / f"{target.stem}.assets"
    checkpoint = work_root / "checkpoint.json"
    partial_markdown = target.with_name(f"{target.stem}.partial.md")
    partial_sidecar = target.with_name(f"{target.stem}.partial.layout.json")
    return work_root, assets, checkpoint, partial_markdown, partial_sidecar


def _source_signature(source: Path) -> dict[str, Any]:
    stat = source.stat()
    return {
        "path": str(source),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def _prepare_layout_workspace(
    source: Path,
    target: Path,
    *,
    pages_total: int,
    settings: dict[str, Any],
    resume_partial: bool,
    overwrite: bool,
) -> tuple[Path, Path, dict[str, Any]]:
    work_root, assets, checkpoint, partial_markdown, partial_sidecar = (
        _layout_workspace_paths(target)
    )
    expected_signature = _source_signature(source)
    if work_root.exists() and overwrite:
        shutil.rmtree(work_root)
        partial_markdown.unlink(missing_ok=True)
        partial_sidecar.unlink(missing_ok=True)
    if work_root.exists():
        if not resume_partial:
            raise ValueError(
                f"Partial OCR checkpoint exists for {source}; enable "
                "resume_partial or use overwrite=True"
            )
        try:
            layout = json.loads(checkpoint.read_text(encoding="utf-8"))
        except Exception as error:
            raise ValueError(
                f"Cannot read partial OCR checkpoint {checkpoint}: {error}; "
                "use overwrite=True to start over"
            ) from error
        if (
            layout.get("source_signature") != expected_signature
            or layout.get("settings") != settings
            or layout.get("pages_total") != pages_total
        ):
            raise ValueError(
                f"Partial OCR checkpoint for {source} is incompatible with the "
                "current source or settings; use overwrite=True to start over"
            )
        assets.mkdir(parents=True, exist_ok=True)
        return work_root, assets, layout

    assets.mkdir(parents=True)
    layout = {
        "schema_version": 2,
        "status": "processing",
        "source": str(source),
        "source_signature": expected_signature,
        "settings": settings,
        "provider": settings["provider"],
        "model": settings["model"],
        "layout_model": settings["layout_model"],
        "dpi": settings["dpi"],
        "pages_total": pages_total,
        "pages": [None] * pages_total,
    }
    _write_layout_checkpoint(target, layout)
    return work_root, assets, layout


def _write_layout_checkpoint(target: Path, layout: dict[str, Any]) -> None:
    work_root, _, checkpoint, partial_markdown, partial_sidecar = (
        _layout_workspace_paths(target)
    )
    asset_prefix = f"{work_root.name}/"
    snapshot = copy.deepcopy(layout)
    snapshot["status"] = "processing"
    _atomic_write_json(checkpoint, snapshot)
    partial_snapshot = copy.deepcopy(snapshot)
    partial_snapshot["asset_prefix"] = asset_prefix
    _atomic_write_json(partial_sidecar, partial_snapshot)
    _atomic_write_text(
        partial_markdown,
        _render_layout_document(snapshot, partial=True, asset_prefix=asset_prefix),
    )


def _clone_layout_assets(source: Path, destination: Path) -> None:
    try:
        shutil.copytree(source, destination, copy_function=os.link)
    except OSError:
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination, copy_function=shutil.copy2)


def _publish_completed_layout(
    target: Path,
    layout: dict[str, Any],
) -> tuple[str, int, int, int]:
    work_root, assets, _, partial_markdown, partial_sidecar = _layout_workspace_paths(
        target
    )
    regions_total, regions_completed, regions_failed = _layout_counts(layout)
    status = "partial" if regions_failed else "processed"
    final_layout = copy.deepcopy(layout)
    final_layout["status"] = status
    markdown = _render_layout_document(final_layout, partial=False)
    with TemporaryDirectory(
        dir=target.parent, prefix=f".{target.stem}.layout-publish-"
    ) as temporary_directory:
        stage_root = Path(temporary_directory)
        _clone_layout_assets(assets, stage_root / assets.name)
        _publish_layout_bundle(stage_root, target, markdown, final_layout)
    shutil.rmtree(work_root)
    partial_markdown.unlink(missing_ok=True)
    partial_sidecar.unlink(missing_ok=True)
    return status, regions_total, regions_completed, regions_failed


def _layout_settings(
    *,
    provider: OCRProvider,
    model: str,
    prompt: str | None,
    dpi: int | None,
    layout_model: str,
    layout_threshold: float,
    embed_page_image: bool,
    max_output_tokens: int,
) -> dict[str, Any]:
    return {
        "provider": provider,
        "model": model,
        "prompt": prompt,
        "dpi": dpi,
        "layout_model": layout_model,
        "layout_threshold": layout_threshold,
        "embed_page_image": embed_page_image,
        "max_output_tokens": max_output_tokens,
        "response_validation_version": _LAYOUT_RESPONSE_VALIDATION_VERSION,
    }


@contextmanager
def _exclusive_output_lock(lock_path: Path, description: str) -> Iterator[None]:
    lock_path.parent.mkdir(parents=True, exist_ok=True)
    with lock_path.open("a+", encoding="utf-8") as lock_file:
        try:
            fcntl.flock(lock_file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            lock_file.seek(0)
            owner = lock_file.read().strip() or "unknown process"
            raise RuntimeError(
                f"Another OCR process holds the {description} lock: {owner}"
            ) from error
        lock_file.seek(0)
        lock_file.truncate()
        lock_file.write(
            json.dumps({"pid": os.getpid(), "description": description}) + "\n"
        )
        lock_file.flush()
        try:
            yield
        finally:
            fcntl.flock(lock_file.fileno(), fcntl.LOCK_UN)


def _atomic_write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path: Path | None = None
    try:
        with NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="\n",
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
            delete=False,
        ) as temporary_file:
            temporary_file.write(content)
            temporary_path = Path(temporary_file.name)
        temporary_path.replace(path)
        temporary_path = None
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


def _atomic_write_json(path: Path, content: dict[str, Any]) -> None:
    _atomic_write_text(
        path,
        json.dumps(content, ensure_ascii=False, indent=2) + "\n",
    )


In [ ]:
# | export
def ocr_pdf(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _OCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
    layout_max_output_tokens: int = _DEFAULT_LAYOUT_MAX_OUTPUT_TOKENS,
) -> OCRResult:
    """Convert one PDF, checkpointing structured OCR until final publication."""
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if layout_max_output_tokens <= 0:
        raise ValueError("layout_max_output_tokens must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        lock_path = target.parent / f".{target.stem}.ocr.lock"
        with _exclusive_output_lock(lock_path, str(target)):
            if not source.is_file():
                raise FileNotFoundError(f"PDF does not exist: {source}")
            detector = layout_detector
            if selected_layout_mode == "pp-doclayout" and detector is None:
                detector = create_pp_doclayout_detector(
                    model_name=layout_model,
                    threshold=layout_threshold,
                    device=layout_device,
                )
                owned_detector = detector
            with pymupdf.open(source) as document:
                if document.needs_pass:
                    raise ValueError("PDF requires a password")
                pages_total = document.page_count
                if pages_total == 0:
                    raise ValueError("PDF contains no pages")

                if selected_layout_mode == "plain":
                    page_sections: list[str] = []
                    for page_number, page in enumerate(document, start=1):
                        page_sections.append(
                            _page_markdown(
                                page,
                                page_number,
                                client=client,
                                provider=selected_provider,
                                model=selected_model,
                                prompt=selected_prompt,
                                dpi=dpi,
                            )
                        )
                        pages_completed = page_number
                    markdown = "\n\n".join(page_sections).rstrip() + "\n"
                    _atomic_write_text(target, markdown)
                    return OCRResult(
                        source, target, "processed", pages_total, pages_completed
                    )

                assert detector is not None
                target.parent.mkdir(parents=True, exist_ok=True)
                settings = _layout_settings(
                    provider=selected_provider,
                    model=selected_model,
                    prompt=prompt,
                    dpi=dpi,
                    layout_model=layout_model,
                    layout_threshold=layout_threshold,
                    embed_page_image=embed_page_image,
                    max_output_tokens=layout_max_output_tokens,
                )
                _, stage_assets, layout = _prepare_layout_workspace(
                    source,
                    target,
                    pages_total=pages_total,
                    settings=settings,
                    resume_partial=resume_partial,
                    overwrite=overwrite,
                )

                def checkpoint(page_record: dict[str, Any]) -> None:
                    layout["pages"][int(page_record["page_number"]) - 1] = (
                        copy.deepcopy(page_record)
                    )
                    _write_layout_checkpoint(target, layout)

                for page_number, page in enumerate(document, start=1):
                    existing_record = layout["pages"][page_number - 1]
                    if (
                        isinstance(existing_record, dict)
                        and existing_record.get("status") == "completed"
                    ):
                        pages_completed += 1
                        continue
                    _, page_record = _layout_page_sync(
                        page.get_pixmap(dpi=dpi, alpha=False),
                        page_number,
                        target=target,
                        stage_assets=stage_assets,
                        detector=detector,
                        client=client,
                        provider=selected_provider,
                        model=selected_model,
                        prompt=prompt,
                        embed_page_image=embed_page_image,
                        max_output_tokens=layout_max_output_tokens,
                        existing_record=(
                            existing_record
                            if isinstance(existing_record, dict)
                            else None
                        ),
                        checkpoint=checkpoint,
                    )
                    layout["pages"][page_number - 1] = page_record
                    pages_completed += 1
                status, regions_total, regions_completed, regions_failed = (
                    _publish_completed_layout(target, layout)
                )
                error = (
                    f"{regions_failed} layout regions failed; see warnings and sidecar"
                    if regions_failed
                    else None
                )
                return OCRResult(
                    source,
                    target,
                    cast(Literal["processed", "partial"], status),
                    pages_total,
                    pages_completed,
                    error,
                    regions_total,
                    regions_completed,
                    regions_failed,
                )
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                owned_detector.stop()
            except Exception:
                pass


In [ ]:
# | export
def ocr_image(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _OCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    overwrite: bool = False,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
    layout_max_output_tokens: int = _DEFAULT_LAYOUT_MAX_OUTPUT_TOKENS,
) -> OCRResult:
    """Convert one image to Markdown, optionally with PP-DocLayout-V3 regions."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    if layout_max_output_tokens <= 0:
        raise ValueError("layout_max_output_tokens must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        lock_path = target.parent / f".{target.stem}.ocr.lock"
        with _exclusive_output_lock(lock_path, str(target)):
            if not source.is_file():
                raise FileNotFoundError(f"Image does not exist: {source}")
            if source.suffix.casefold() not in _IMAGE_SUFFIXES:
                raise ValueError(
                    f"Unsupported image type: {source.suffix or '<none>'}"
                )

            pixmap = _load_image_pixmap(source)
            pages_total = 1
            if selected_layout_mode == "plain":
                markdown = (
                    _pixmap_markdown(
                        pixmap,
                        "Image",
                        client=client,
                        provider=selected_provider,
                        model=selected_model,
                        prompt=selected_prompt,
                    ).markdown.rstrip()
                    + "\n"
                )
                _atomic_write_text(target, markdown)
                pages_completed = 1
                return OCRResult(
                    source, target, "processed", pages_total, pages_completed
                )

            detector = layout_detector
            if detector is None:
                detector = create_pp_doclayout_detector(
                    model_name=layout_model,
                    threshold=layout_threshold,
                    device=layout_device,
                )
                owned_detector = detector
            target.parent.mkdir(parents=True, exist_ok=True)
            settings = _layout_settings(
                provider=selected_provider,
                model=selected_model,
                prompt=prompt,
                dpi=None,
                layout_model=layout_model,
                layout_threshold=layout_threshold,
                embed_page_image=embed_page_image,
                max_output_tokens=layout_max_output_tokens,
            )
            _, stage_assets, layout = _prepare_layout_workspace(
                source,
                target,
                pages_total=1,
                settings=settings,
                resume_partial=resume_partial,
                overwrite=overwrite,
            )

            def checkpoint(page_record: dict[str, Any]) -> None:
                layout["pages"][0] = copy.deepcopy(page_record)
                _write_layout_checkpoint(target, layout)

            existing_record = layout["pages"][0]
            if not (
                isinstance(existing_record, dict)
                and existing_record.get("status") == "completed"
            ):
                _, page_record = _layout_page_sync(
                    pixmap,
                    1,
                    target=target,
                    stage_assets=stage_assets,
                    detector=detector,
                    client=client,
                    provider=selected_provider,
                    model=selected_model,
                    prompt=prompt,
                    embed_page_image=embed_page_image,
                    max_output_tokens=layout_max_output_tokens,
                    existing_record=(
                        existing_record
                        if isinstance(existing_record, dict)
                        else None
                    ),
                    checkpoint=checkpoint,
                )
                layout["pages"][0] = page_record
            pages_completed = 1
            status, regions_total, regions_completed, regions_failed = (
                _publish_completed_layout(target, layout)
            )
            error = (
                f"{regions_failed} layout regions failed; see warnings and sidecar"
                if regions_failed
                else None
            )
            return OCRResult(
                source,
                target,
                cast(Literal["processed", "partial"], status),
                pages_total,
                pages_completed,
                error,
                regions_total,
                regions_completed,
                regions_failed,
            )
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                owned_detector.stop()
            except Exception:
                pass


In [ ]:
# | export
async def _ocr_pdf_async(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
    page_concurrency: int = 1,
    request_semaphore: asyncio.Semaphore | None = None,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
    layout_max_output_tokens: int = _DEFAULT_LAYOUT_MAX_OUTPUT_TOKENS,
    layout_semaphore: asyncio.Semaphore | None = None,
    page_started: Callable[[int], None] | None = None,
    page_progress: Callable[[int, int, float], None] | None = None,
) -> OCRResult:
    """Asynchronously convert one PDF while preserving page order."""
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")
    if layout_max_output_tokens <= 0:
        raise ValueError("layout_max_output_tokens must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        detector = layout_detector
        stage_assets: Path | None = None
        layout: dict[str, Any] | None = None
        if selected_layout_mode == "pp-doclayout":
            if detector is None:
                detector = await asyncio.to_thread(
                    create_pp_doclayout_detector,
                    model_name=layout_model,
                    threshold=layout_threshold,
                    device=layout_device,
                )
                owned_detector = detector
            target.parent.mkdir(parents=True, exist_ok=True)
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")
            if page_started is not None:
                page_started(pages_total)

            checkpoint_lock = asyncio.Lock()
            if selected_layout_mode == "pp-doclayout":
                settings = _layout_settings(
                    provider=selected_provider,
                    model=selected_model,
                    prompt=prompt,
                    dpi=dpi,
                    layout_model=layout_model,
                    layout_threshold=layout_threshold,
                    embed_page_image=embed_page_image,
                    max_output_tokens=layout_max_output_tokens,
                )
                _, stage_assets, layout = _prepare_layout_workspace(
                    source,
                    target,
                    pages_total=pages_total,
                    settings=settings,
                    resume_partial=resume_partial,
                    overwrite=overwrite,
                )

                async def checkpoint_page(page_record: dict[str, Any]) -> None:
                    assert layout is not None
                    async with checkpoint_lock:
                        layout["pages"][int(page_record["page_number"]) - 1] = (
                            copy.deepcopy(page_record)
                        )
                        snapshot = copy.deepcopy(layout)
                        await asyncio.to_thread(
                            _write_layout_checkpoint, target, snapshot
                        )
            else:
                checkpoint_page = None

            page_sections = [""] * pages_total
            page_records: list[dict[str, Any] | None] = (
                list(layout["pages"]) if layout is not None else [None] * pages_total
            )
            page_tasks: set[
                asyncio.Task[tuple[int, str, dict[str, Any] | None, float]]
            ] = set()
            pending: set[
                asyncio.Task[tuple[int, str, dict[str, Any] | None, float]]
            ] = set()
            next_page_number = 1

            async def process_page(
                page_number: int,
            ) -> tuple[int, str, dict[str, Any] | None, float]:
                page_started_at = perf_counter()
                existing_record = page_records[page_number - 1]
                if (
                    selected_layout_mode == "pp-doclayout"
                    and isinstance(existing_record, dict)
                    and existing_record.get("status") == "completed"
                ):
                    return (
                        page_number,
                        _render_layout_page(existing_record, page_number),
                        existing_record,
                        perf_counter() - page_started_at,
                    )
                page = document.load_page(page_number - 1)
                if selected_layout_mode == "plain":
                    section = await _page_markdown_async(
                        page,
                        page_number,
                        client=client,
                        provider=selected_provider,
                        model=selected_model,
                        prompt=selected_prompt,
                        dpi=dpi,
                        request_semaphore=request_semaphore,
                    )
                    page_record = None
                else:
                    assert (
                        detector is not None
                        and stage_assets is not None
                        and checkpoint_page is not None
                    )
                    section, page_record = await _layout_page_async(
                        page.get_pixmap(dpi=dpi, alpha=False),
                        page_number,
                        target=target,
                        stage_assets=stage_assets,
                        detector=detector,
                        client=client,
                        provider=selected_provider,
                        model=selected_model,
                        prompt=prompt,
                        embed_page_image=embed_page_image,
                        request_semaphore=request_semaphore,
                        layout_semaphore=layout_semaphore,
                        max_output_tokens=layout_max_output_tokens,
                        existing_record=(
                            existing_record
                            if isinstance(existing_record, dict)
                            else None
                        ),
                        checkpoint=checkpoint_page,
                    )
                return (
                    page_number,
                    section,
                    page_record,
                    perf_counter() - page_started_at,
                )

            def schedule_next_page() -> None:
                nonlocal next_page_number
                task = asyncio.create_task(process_page(next_page_number))
                page_tasks.add(task)
                pending.add(task)
                next_page_number += 1

            for _ in range(min(page_concurrency, pages_total)):
                schedule_next_page()

            try:
                while pending:
                    completed, pending = await asyncio.wait(
                        pending, return_when=asyncio.FIRST_COMPLETED
                    )
                    for page_task in completed:
                        page_number, section, page_record, elapsed_s = (
                            page_task.result()
                        )
                        page_sections[page_number - 1] = section
                        page_records[page_number - 1] = page_record
                        pages_completed += 1
                        if page_progress is not None:
                            page_progress(pages_completed, pages_total, elapsed_s)
                        if next_page_number <= pages_total:
                            schedule_next_page()
            finally:
                for page_task in page_tasks:
                    if not page_task.done():
                        page_task.cancel()
                await asyncio.gather(*page_tasks, return_exceptions=True)

        if selected_layout_mode == "plain":
            markdown = "\n\n".join(page_sections).rstrip() + "\n"
            _atomic_write_text(target, markdown)
            return OCRResult(
                source, target, "processed", pages_total, pages_completed
            )

        assert layout is not None
        layout["pages"] = [
            copy.deepcopy(record) if isinstance(record, dict) else None
            for record in page_records
        ]
        status, regions_total, regions_completed, regions_failed = (
            await asyncio.to_thread(
                _publish_completed_layout,
                target,
                layout,
            )
        )
        error = (
            f"{regions_failed} layout regions failed; see warnings and sidecar"
            if regions_failed
            else None
        )
        return OCRResult(
            source,
            target,
            cast(Literal["processed", "partial"], status),
            pages_total,
            pages_completed,
            error,
            regions_total,
            regions_completed,
            regions_failed,
        )
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                await asyncio.to_thread(owned_detector.stop)
            except Exception:
                pass


async def _ocr_image_async(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: _AsyncOCRClient,
    provider: OCRProvider = "ollama",
    model: str | None = None,
    prompt: str | None = None,
    overwrite: bool = False,
    request_semaphore: asyncio.Semaphore | None = None,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
    layout_max_output_tokens: int = _DEFAULT_LAYOUT_MAX_OUTPUT_TOKENS,
    layout_semaphore: asyncio.Semaphore | None = None,
) -> OCRResult:
    """Asynchronously convert one PNG or JPEG image."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_provider = _resolve_provider(provider)
    selected_model = _resolve_model(selected_provider, model)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_prompt = _resolve_prompt(selected_provider, prompt)
    if layout_max_output_tokens <= 0:
        raise ValueError("layout_max_output_tokens must be greater than zero")

    if target.exists():
        if not target.is_file():
            return OCRResult(
                source, target, "failed", error="Markdown target is not a file"
            )
        if not overwrite:
            return OCRResult(source, target, "skipped")

    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")

        pixmap = _load_image_pixmap(source)
        pages_total = 1
        if selected_layout_mode == "plain":
            response = await _pixmap_markdown_async(
                    pixmap,
                    "Image",
                    client=client,
                    provider=selected_provider,
                    model=selected_model,
                    prompt=selected_prompt,
                    request_semaphore=request_semaphore,
                )
            markdown = response.markdown.rstrip() + "\n"
            _atomic_write_text(target, markdown)
            pages_completed = 1
            return OCRResult(
                source, target, "processed", pages_total, pages_completed
            )

        detector = layout_detector
        if detector is None:
            detector = await asyncio.to_thread(
                create_pp_doclayout_detector,
                model_name=layout_model,
                threshold=layout_threshold,
                device=layout_device,
            )
            owned_detector = detector
        target.parent.mkdir(parents=True, exist_ok=True)
        settings = _layout_settings(
            provider=selected_provider,
            model=selected_model,
            prompt=prompt,
            dpi=None,
            layout_model=layout_model,
            layout_threshold=layout_threshold,
            embed_page_image=embed_page_image,
            max_output_tokens=layout_max_output_tokens,
        )
        _, stage_assets, layout = _prepare_layout_workspace(
            source,
            target,
            pages_total=1,
            settings=settings,
            resume_partial=resume_partial,
            overwrite=overwrite,
        )
        checkpoint_lock = asyncio.Lock()

        async def checkpoint_page(page_record: dict[str, Any]) -> None:
            async with checkpoint_lock:
                layout["pages"][0] = copy.deepcopy(page_record)
                snapshot = copy.deepcopy(layout)
                await asyncio.to_thread(_write_layout_checkpoint, target, snapshot)

        existing_record = layout["pages"][0]
        if not (
            isinstance(existing_record, dict)
            and existing_record.get("status") == "completed"
        ):
            _, page_record = await _layout_page_async(
                pixmap,
                1,
                target=target,
                stage_assets=stage_assets,
                detector=detector,
                client=client,
                provider=selected_provider,
                model=selected_model,
                prompt=prompt,
                embed_page_image=embed_page_image,
                request_semaphore=request_semaphore,
                layout_semaphore=layout_semaphore,
                max_output_tokens=layout_max_output_tokens,
                existing_record=(
                    existing_record if isinstance(existing_record, dict) else None
                ),
                checkpoint=checkpoint_page,
            )
            layout["pages"][0] = page_record
        pages_completed = 1
        status, regions_total, regions_completed, regions_failed = (
            await asyncio.to_thread(_publish_completed_layout, target, layout)
        )
        error = (
            f"{regions_failed} layout regions failed; see warnings and sidecar"
            if regions_failed
            else None
        )
        return OCRResult(
            source,
            target,
            cast(Literal["processed", "partial"], status),
            pages_total,
            pages_completed,
            error,
            regions_total,
            regions_completed,
            regions_failed,
        )
    except Exception as error:
        return OCRResult(
            source,
            target,
            "failed",
            pages_total,
            pages_completed,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                await asyncio.to_thread(owned_detector.stop)
            except Exception:
                pass


In [ ]:
# | export
def _create_async_ocr_client(
    provider: OCRProvider,
    *,
    host: str | Sequence[str],
    request_timeout_s: float,
    max_concurrency: int = 4,
) -> _AsyncOCRClient:
    if provider == "ollama":
        hosts = _normalize_ollama_hosts(host)
        if len(hosts) > 1:
            return _AsyncOllamaPool(
                hosts,
                request_timeout_s=request_timeout_s,
                max_concurrency=max_concurrency,
            )
        return AsyncOllamaClient(host=hosts[0], timeout=request_timeout_s)

    api_key = os.getenv("DASHSCOPE_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            f"DASHSCOPE_API_KEY is not configured in {PROJ_ROOT / '.env'}"
        )
    base_url = os.getenv("DASHSCOPE_API_URL", "").strip() or _DEFAULT_DASHSCOPE_BASE_URL
    if (
        not base_url.startswith(("http://", "https://"))
        or "/compatible-mode/" not in base_url
    ):
        raise RuntimeError(
            "DASHSCOPE_API_URL must be an OpenAI-compatible HTTP(S) endpoint"
        )
    return AsyncOpenAI(api_key=api_key, base_url=base_url, timeout=request_timeout_s)


@dataclass
class _ActivePageProgress:
    source_path: Path
    pages_total: int | None = None
    pages_completed: int = 0
    last_page_elapsed_s: float | None = None


def _running_in_notebook() -> bool:
    """Return whether output is being rendered by a Jupyter kernel."""
    try:
        from IPython import get_ipython
    except ImportError:
        return False
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"


class _OCRFolderProgress:
    """Render page and file progress reliably in terminals and notebooks."""

    def __init__(self, total_files: int, description: str, show_pages: bool) -> None:
        self._total_files = total_files
        self._files_completed = 0
        self._description = description
        self._show_pages = show_pages
        self._active_order: list[int] = []
        self._active: dict[int, _ActivePageProgress] = {}
        self._displayed_index: int | None = None
        self._notebook_mode = _running_in_notebook()
        self._html_factory: Callable[[str], Any] | None = None
        self._display: Callable[..., Any] | None = None
        self._display_handle: Any = None
        self._page_bar: Any = None
        self._file_bar: Any = None

        if self._notebook_mode:
            from IPython.display import HTML, display

            self._html_factory = HTML
            self._display = display
            self._refresh_notebook()
        else:
            self._page_bar = (
                tqdm(
                    total=None,
                    desc="OCR pages: waiting",
                    unit="page",
                    position=0,
                    leave=False,
                    dynamic_ncols=True,
                )
                if show_pages
                else None
            )
            self._file_bar = tqdm(
                total=total_files,
                desc=description,
                unit="file",
                position=1 if show_pages else 0,
                leave=True,
                dynamic_ncols=True,
            )

    def start_file(
        self, index: int, source_path: Path, pages_total: int | None = None
    ) -> None:
        self._active[index] = _ActivePageProgress(source_path, pages_total)
        self._active_order.append(index)
        self._refresh_page_bar()

    def set_pages_total(self, index: int, pages_total: int) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            self._refresh_page_bar()

    def complete_page(
        self, index: int, pages_completed: int, pages_total: int, elapsed_s: float
    ) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            state.pages_completed = pages_completed
            state.last_page_elapsed_s = elapsed_s
            self._refresh_page_bar()

    def finish_file(self, index: int) -> None:
        self._active.pop(index, None)
        if index in self._active_order:
            self._active_order.remove(index)
        self._files_completed += 1
        if self._file_bar is not None:
            self._file_bar.update(1)
        self._refresh_page_bar()

    def _refresh_page_bar(self) -> None:
        if self._notebook_mode:
            self._refresh_notebook()
            return
        if self._page_bar is None:
            return
        if not self._active_order:
            self._displayed_index = None
            self._page_bar.clear()
            self._page_bar.set_description_str("OCR pages: waiting", refresh=False)
            self._page_bar.total = None
            self._page_bar.n = 0
            self._page_bar.set_postfix_str("", refresh=False)
            self._page_bar.refresh()
            return

        newest_index = self._active_order[-1]
        state = self._active[newest_index]
        if newest_index != self._displayed_index:
            self._page_bar.clear()
            self._displayed_index = newest_index
            self._page_bar.n = 0
            self._page_bar.last_print_n = 0
            now = self._page_bar._time()
            self._page_bar.start_t = now
            self._page_bar.last_print_t = now
        self._page_bar.set_description_str(
            f"OCR pages: {state.source_path}", refresh=False
        )
        self._page_bar.total = state.pages_total
        self._page_bar.n = state.pages_completed
        elapsed = state.last_page_elapsed_s
        self._page_bar.set_postfix_str(
            f"last={elapsed:.2f}s" if elapsed is not None else "",
            refresh=False,
        )
        self._page_bar.refresh()

    @staticmethod
    def _html_progress_row(
        label: str, completed: int, total: int | None, detail: str = ""
    ) -> str:
        count = f"{completed}/{total}" if total is not None else f"{completed}/?"
        percentage = 100 * completed / total if total else 0
        progress = (
            f'<progress value="{min(completed, total)}" max="{total}" '
            'style="width:100%;height:0.8rem"></progress>'
            if total is not None
            else '<progress style="width:100%;height:0.8rem"></progress>'
        )
        suffix = f" &nbsp; {escape(detail)}" if detail else ""
        return (
            '<div style="margin:0 0 0.45rem 0">'
            '<div style="display:flex;gap:1rem;justify-content:space-between;"'
            ">"
            f'<span style="overflow-wrap:anywhere">{escape(label)}</span>'
            f'<span style="white-space:nowrap">{percentage:.0f}% &nbsp; '
            f"{count}{suffix}</span></div>{progress}</div>"
        )

    def _notebook_html(self) -> str:
        rows: list[str] = []
        if self._show_pages:
            if self._active_order:
                state = self._active[self._active_order[-1]]
                elapsed = state.last_page_elapsed_s
                detail = f"last={elapsed:.2f}s" if elapsed is not None else ""
                rows.append(
                    self._html_progress_row(
                        f"OCR pages: {state.source_path}",
                        state.pages_completed,
                        state.pages_total,
                        detail,
                    )
                )
            else:
                rows.append(self._html_progress_row("OCR pages: waiting", 0, None))
        rows.append(
            self._html_progress_row(
                self._description, self._files_completed, self._total_files
            )
        )
        return (
            '<div style="font-family:var(--vscode-editor-font-family,monospace);'
            'font-size:var(--vscode-editor-font-size,13px);padding:0.25rem 0">'
            + "".join(rows)
            + "</div>"
        )

    def _refresh_notebook(self) -> None:
        if self._html_factory is None or self._display is None:
            return
        content = self._html_factory(self._notebook_html())
        if self._display_handle is None:
            self._display_handle = self._display(content, display_id=True)
        else:
            self._display_handle.update(content)

    def close(self) -> None:
        if self._notebook_mode:
            self._refresh_notebook()
            return
        if self._page_bar is not None:
            self._page_bar.close()
        if self._file_bar is not None:
            self._file_bar.close()


def _print_ocr_file_report(results: Sequence[OCRResult]) -> None:
    """Print every discovered source by modification time, newest first."""
    dated_results: list[tuple[int, OCRResult]] = []
    for result in results:
        try:
            modified_ns = result.source_path.stat().st_mtime_ns
        except OSError:
            modified_ns = -1
        dated_results.append((modified_ns, result))
    dated_results.sort(
        key=lambda item: (
            -item[0],
            item[1].source_path.as_posix().casefold(),
        )
    )

    print(f"OCR files (newest to oldest, {len(dated_results)} total):")
    for modified_ns, result in dated_results:
        modified = (
            datetime.fromtimestamp(modified_ns / 1_000_000_000)
            .astimezone()
            .isoformat(timespec="seconds")
            if modified_ns >= 0
            else "unknown"
        )
        print(
            f"OCR file: {result.source_path} | status={result.status} | "
            f"modified={modified}"
        )


async def ocr_folder(
    root_folder: Path | str,
    *,
    provider: OCRProvider = "ollama",
    host: str | Sequence[str] = "http://127.0.0.1:11434",
    model: str | None = None,
    prompt: str | None = None,
    dpi: int = 200,
    overwrite: bool = False,
    request_timeout_s: float = 180,
    max_concurrency: int = 4,
    page_concurrency: int = 1,
    show_page_progress: bool = True,
    client: _AsyncOCRClient | None = None,
    layout_mode: OCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
    layout_max_output_tokens: int = _DEFAULT_LAYOUT_MAX_OUTPUT_TOKENS,
) -> list[OCRResult]:
    """Concurrently OCR PDFs and images under ``root_folder``.

    At most ``max_concurrency`` files and OCR requests are active. Each PDF
    may process up to ``page_concurrency`` pages concurrently, while returned
    results and Markdown sections preserve deterministic source/page order.
    Existing outputs are omitted from live progress. When page progress is shown,
    the newest active file is displayed above aggregate file progress.
    The final report includes every result, sorted by source modification time
    from newest to oldest.
    For Ollama, ``host`` may be one URL or a sequence of URLs; multi-host
    requests use a preflighted, availability-driven pool.
    """
    selected_provider = _resolve_provider(provider)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    selected_model = _resolve_model(selected_provider, model)
    _resolve_prompt(selected_provider, prompt)
    root = _resolve_root(root_folder)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")
    if max_concurrency <= 0:
        raise ValueError("max_concurrency must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")
    if layout_max_output_tokens <= 0:
        raise ValueError("layout_max_output_tokens must be greater than zero")
    if not 0 <= layout_threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    if selected_layout_mode == "pp-doclayout" and not layout_model.strip():
        raise ValueError("layout_model must not be empty")

    jobs = _ocr_jobs(root)
    if not jobs:
        print(f"No PDF, PNG, or JPEG files found under {root}")
        return []

    output_root = root / ".md"
    legacy_workspaces = sorted(
        (
            path
            for path in output_root.rglob(".*.layout-*")
            if path.is_dir()
            and not path.name.endswith(".layout-work")
            and ".layout-publish-" not in path.name
        ),
        key=lambda path: path.as_posix().casefold(),
    )
    if legacy_workspaces:
        print(
            f"Found {len(legacy_workspaces)} legacy OCR staging directories; "
            "they were left untouched because an older kernel may still own them."
        )
        for legacy_workspace in legacy_workspaces[:5]:
            print(f"Legacy OCR staging: {legacy_workspace}")
        if len(legacy_workspaces) > 5:
            print(f"... and {len(legacy_workspaces) - 5} more")

    results_by_index: dict[int, OCRResult] = {}
    pending_jobs: list[tuple[int, Path, Path]] = []
    for index, (source_path, markdown_path) in enumerate(jobs):
        if not overwrite and markdown_path.is_file():
            results_by_index[index] = OCRResult(source_path, markdown_path, "skipped")
        else:
            pending_jobs.append((index, source_path, markdown_path))

    if not pending_jobs:
        results = [results_by_index[index] for index in range(len(jobs))]
        print(
            f"OCR complete: 0 processed, 0 partial, "
            f"{len(results)} skipped, 0 failed"
        )
        _print_ocr_file_report(results)
        return results

    output_lock = _exclusive_output_lock(
        root / ".ocr-folder.lock", f"OCR output tree {root / '.md'}"
    )
    output_lock.__enter__()
    try:
        ocr_client = (
            client
            if client is not None
            else _create_async_ocr_client(
                selected_provider,
                host=host,
                request_timeout_s=request_timeout_s,
                max_concurrency=max_concurrency,
            )
        )
    except BaseException:
        output_lock.__exit__(None, None, None)
        raise
    if selected_provider == "ollama":
        try:
            await ocr_client.show(selected_model)  # type: ignore[union-attr]
        except BaseException as error:
            if client is None:
                await ocr_client.close()
            output_lock.__exit__(None, None, None)
            if isinstance(error, asyncio.CancelledError):
                raise
            raise RuntimeError(
                f"Cannot use Ollama model '{selected_model}' at {host}. "
                f"Ensure Ollama is running and run `ollama pull {selected_model}`. "
                f"Original error: {error}"
            ) from error
        if isinstance(ocr_client, _AsyncOllamaPool):
            print(
                "Ollama host pool: "
                f"{len(ocr_client.active_hosts)}/{len(ocr_client.configured_hosts)} "
                f"hosts active, {ocr_client.request_slots} request slots "
                f"({ocr_client.slots_per_host}/host)"
            )
            for unavailable_host, error in ocr_client.unavailable_hosts:
                print(f"Ollama host unavailable: {unavailable_host} ({error})")

    active_layout_detector = layout_detector
    owned_layout_detector = False
    if selected_layout_mode == "pp-doclayout" and active_layout_detector is None:
        try:
            active_layout_detector = await asyncio.to_thread(
                create_pp_doclayout_detector,
                model_name=layout_model,
                threshold=layout_threshold,
                device=layout_device,
            )
            owned_layout_detector = True
        except BaseException:
            if client is None:
                await ocr_client.close()
            output_lock.__exit__(None, None, None)
            raise

    description = (
        f"OCR files ({selected_provider}, layout={selected_layout_mode}, "
        f"files/requests={max_concurrency}, "
        f"pages/PDF={page_concurrency})"
    )
    progress = _OCRFolderProgress(len(pending_jobs), description, show_page_progress)
    file_semaphore = asyncio.Semaphore(max_concurrency)
    request_semaphore = asyncio.Semaphore(max_concurrency)
    layout_semaphore = asyncio.Semaphore(1)

    async def run_job(
        index: int, source_path: Path, markdown_path: Path
    ) -> tuple[int, OCRResult, float]:
        async with file_semaphore:
            started_at = perf_counter()
            is_pdf = source_path.suffix.casefold() == ".pdf"
            progress.start_file(index, source_path, None if is_pdf else 1)

            def report_pages_total(pages_total: int) -> None:
                progress.set_pages_total(index, pages_total)

            def report_page(
                pages_completed: int, pages_total: int, elapsed_s: float
            ) -> None:
                progress.complete_page(index, pages_completed, pages_total, elapsed_s)

            common_arguments = {
                "client": ocr_client,
                "provider": selected_provider,
                "model": selected_model,
                "prompt": prompt,
                "overwrite": overwrite,
                "request_semaphore": request_semaphore,
                "layout_mode": selected_layout_mode,
                "layout_detector": active_layout_detector,
                "layout_model": layout_model,
                "layout_threshold": layout_threshold,
                "layout_device": layout_device,
                "embed_page_image": embed_page_image,
                "resume_partial": resume_partial,
                "layout_max_output_tokens": layout_max_output_tokens,
                "layout_semaphore": layout_semaphore,
            }
            try:
                if is_pdf:
                    result = await _ocr_pdf_async(
                        source_path,
                        markdown_path,
                        dpi=dpi,
                        page_concurrency=page_concurrency,
                        page_started=(
                            report_pages_total if show_page_progress else None
                        ),
                        page_progress=report_page if show_page_progress else None,
                        **common_arguments,
                    )
                else:
                    result = await _ocr_image_async(
                        source_path,
                        markdown_path,
                        **common_arguments,
                    )
                    if result.status in {"processed", "partial"} and show_page_progress:
                        progress.complete_page(index, 1, 1, perf_counter() - started_at)
            except Exception as error:
                result = OCRResult(
                    source_path,
                    markdown_path,
                    "failed",
                    error=f"{type(error).__name__}: {error}",
                )
            finally:
                progress.finish_file(index)
            return index, result, perf_counter() - started_at

    tasks = [
        asyncio.create_task(run_job(index, source_path, markdown_path))
        for index, source_path, markdown_path in pending_jobs
    ]
    completion_records: list[str] = []
    cancelled_error: asyncio.CancelledError | None = None

    def remember_result(index: int, result: OCRResult, elapsed_s: float) -> None:
        if index in results_by_index:
            return
        results_by_index[index] = result
        if result.status != "skipped":
            error_text = f" | error={result.error}" if result.error else ""
            completion_records.append(
                f"OCR task: {result.source_path} | status={result.status} | "
                f"elapsed={elapsed_s:.2f}s{error_text}"
            )

    try:
        for completed_task in asyncio.as_completed(tasks):
            index, result, elapsed_s = await completed_task
            remember_result(index, result, elapsed_s)
    except asyncio.CancelledError as error:
        cancelled_error = error
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        for task in tasks:
            if task.cancelled():
                continue
            try:
                index, result, elapsed_s = task.result()
            except BaseException:
                continue
            remember_result(index, result, elapsed_s)
        progress.close()
        if owned_layout_detector and active_layout_detector is not None:
            try:
                await asyncio.to_thread(active_layout_detector.stop)
            except Exception:
                pass
        if client is None:
            await ocr_client.close()
        output_lock.__exit__(None, None, None)

    for record in completion_records:
        print(record)

    if cancelled_error is not None:
        interrupted_results = [
            results_by_index[index] for index in sorted(results_by_index)
        ]
        counts = Counter(result.status for result in interrupted_results)
        print(
            f"OCR interrupted: {counts['processed']} processed, "
            f"{counts['partial']} partial, "
            f"{counts['skipped']} skipped, {counts['failed']} failed"
        )
        _print_ocr_file_report(interrupted_results)
        raise cancelled_error

    results = [results_by_index[index] for index in range(len(jobs))]

    counts = Counter(result.status for result in results)
    print(
        f"OCR complete: {counts['processed']} processed, "
        f"{counts['partial']} partial, "
        f"{counts['skipped']} skipped, {counts['failed']} failed"
    )
    _print_ocr_file_report(results)
    return results

## Configuration and batch execution

Set `OCR_PROVIDER` to `"ollama"` or `"dashscope"`, set `PDF_ROOT` to the root containing PDFs and/or images, then run the final cell with `await`. Set `OCR_LAYOUT_MODE = "pp-doclayout"` for structured reconstruction; leave it as `"plain"` for fast whole-page transcription. Structured mode recognizes text, tables, and formulas, but follows the official GLM-OCR pipeline by preserving figure/chart crops without sending them through `Figure Recognition:`. It writes a durable `<name>.partial.md` and schema-v2 partial sidecar while work is active, checkpoints every completed region, and resumes compatible checkpoints when `RESUME_PARTIAL = True`. Completed bundles contain `.md`, `.layout.json`, and `.assets/`; a failed region produces a linked crop and warning while the result is reported as `partial`. `LAYOUT_MAX_OUTPUT_TOKENS` defaults to 4096 for structured tables and formulas; text regions use the smaller of that value and 1024. Degenerate responses retain only the first stable text unit or first complete HTML table and record status `recovered` plus an audit reason in the sidecar. Unrecoverable output-token truncation remains a failed region. An output-tree lock rejects a second concurrent folder run. Legacy random `.layout-*` directories are not deleted automatically. `LAYOUT_DEVICE = None` auto-selects CUDA when available, while `"cpu"` reserves the GPU for Ollama. `EMBED_PAGE_IMAGE` adds a full-page visual fallback at the cost of output size. The configuration reads `OLLAMA_MAX_CONCURRENCY` as the request-slot count per configured Ollama host and `OLLAMA_PAGE_CONCURRENCY` per PDF, or `OPENAILIKED_MAX_CONCURRENCY` and `OPENAILIKED_PAGE_CONCURRENCY` for DashScope, from `PROJ_ROOT/.env`; existing process-environment values take precedence. The two-host Ollama configuration multiplies the per-host value to derive global `MAX_CONCURRENCY`, while the host pool removes unavailable endpoints and keeps their slots disabled. `PAGE_CONCURRENCY` controls concurrent pages within each PDF. Existing Markdown targets are omitted from live progress and represented as `skipped`; the final file report includes skipped and newly processed sources, sorted by source modification time from newest to oldest. Every Ollama endpoint must have the selected model (normally `glm-ocr`) installed. DashScope reads `DASHSCOPE_API_KEY`, `DASHSCOPE_API_URL`, and optional `OPENAILIKED_OCR_MODEL`; its model defaults to `qwen3.7-plus`. Set `OVERWRITE = True` to regenerate existing malformed output or discard an incompatible checkpoint.

In [ ]:
PDF_ROOT = Path("../res/PDF-20260721")  # Root containing PDFs and/or images.
OCR_PROVIDER: OCRProvider = "ollama"  # Change to "dashscope" for Aliyun.
OLLAMA_HOSTS = [
    "http://127.0.0.1:11434",
    "http://172.27.74.16:11434",
]
OCR_MODEL: str | None = None  # Use the provider or environment default.
OCR_DPI = 200
REQUEST_TIMEOUT_S = 180
OCR_LAYOUT_MODE: OCRLayoutMode = "pp-doclayout" # Use "plain" for no structure.
LAYOUT_MODEL = "PaddlePaddle/PP-DocLayoutV3_safetensors"
LAYOUT_THRESHOLD = 0.3
LAYOUT_DEVICE: str | None = None  # Set to "cpu" to reserve GPU for Ollama.
EMBED_PAGE_IMAGE = False
RESUME_PARTIAL = True
LAYOUT_MAX_OUTPUT_TOKENS = 4096


def _positive_env_int(name: str, default: int) -> int:
    raw_value = os.getenv(name, str(default)).strip()
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be a positive integer, got {raw_value!r}") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero, got {value}")
    return value


_CONCURRENCY_ENV_PREFIX = (
    "OLLAMA" if OCR_PROVIDER == "ollama" else "OPENAILIKED"
)
_CONFIGURED_MAX_CONCURRENCY = _positive_env_int(
    f"{_CONCURRENCY_ENV_PREFIX}_MAX_CONCURRENCY",
    4 if OCR_PROVIDER == "ollama" else 8,
)
MAX_CONCURRENCY = (
    _CONFIGURED_MAX_CONCURRENCY * len(OLLAMA_HOSTS)
    if OCR_PROVIDER == "ollama"
    else _CONFIGURED_MAX_CONCURRENCY
)
PAGE_CONCURRENCY = _positive_env_int(
    f"{_CONCURRENCY_ENV_PREFIX}_PAGE_CONCURRENCY",
    2,
)
SHOW_PAGE_PROGRESS = True
OVERWRITE = False

In [ ]:
# | notest
results = await ocr_folder(
    PDF_ROOT,
    provider=OCR_PROVIDER,
    host=OLLAMA_HOSTS,
    model=OCR_MODEL,
    dpi=OCR_DPI,
    overwrite=OVERWRITE,
    request_timeout_s=REQUEST_TIMEOUT_S,
    max_concurrency=MAX_CONCURRENCY,
    page_concurrency=PAGE_CONCURRENCY,
    show_page_progress=SHOW_PAGE_PROGRESS,
    layout_mode=OCR_LAYOUT_MODE,
    layout_model=LAYOUT_MODEL,
    layout_threshold=LAYOUT_THRESHOLD,
    layout_device=LAYOUT_DEVICE,
    embed_page_image=EMBED_PAGE_IMAGE,
    resume_partial=RESUME_PARTIAL,
    layout_max_output_tokens=LAYOUT_MAX_OUTPUT_TOKENS,
)

# | notest
results

## Tests

The tests use temporary PDFs/images and fake Ollama/OpenAI-compatible clients, so they do not require a running service, incur cloud charges, or write to the repository.

In [ ]:
# | hide
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from types import SimpleNamespace
from unittest.mock import patch

from fastcore.test import test_eq, test_fail


class FakeOllamaClient:
    def __init__(self, responses=(), show_error: Exception | None = None):
        self.responses = list(responses)
        self.show_error = show_error
        self.show_calls = []
        self.chat_calls = []

    def show(self, model):
        self.show_calls.append(model)
        if self.show_error is not None:
            raise self.show_error
        return {}

    def chat(self, **kwargs):
        self.chat_calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat call")
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        return SimpleNamespace(message=SimpleNamespace(content=response))


class FakeOpenAIClient:
    def __init__(self, responses=()):
        self.responses = list(responses)
        self.calls = []
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=response))]
        )


class FakeAsyncOllamaClient:
    def __init__(
        self, responses=(), show_error: Exception | None = None, delay_s=0.01
    ):
        self.responses = list(responses)
        self.show_error = show_error
        self.delay_s = delay_s
        self.show_calls = []
        self.chat_calls = []
        self.active_calls = 0
        self.max_active_calls = 0
        self.closed = False

    async def show(self, model):
        self.show_calls.append(model)
        if self.show_error is not None:
            raise self.show_error
        return {}

    async def chat(self, **kwargs):
        self.chat_calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat call")
        response = self.responses.pop(0)
        delay_s = self.delay_s
        if isinstance(response, tuple):
            delay_s, response = response
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        try:
            await asyncio.sleep(delay_s)
        finally:
            self.active_calls -= 1
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        return SimpleNamespace(message=SimpleNamespace(content=response))

    async def close(self):
        self.closed = True


class FakeAsyncOpenAIClient:
    def __init__(self, responses=(), delay_s=0.01):
        self.responses = list(responses)
        self.delay_s = delay_s
        self.calls = []
        self.active_calls = 0
        self.max_active_calls = 0
        self.closed = False
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    async def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        try:
            await asyncio.sleep(self.delay_s)
        finally:
            self.active_calls -= 1
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=response))]
        )

    async def close(self):
        self.closed = True


class FakeLayoutDetector:
    def __init__(self, regions):
        self.regions = regions
        self.calls = []
        self.stopped = False

    def process(self, images, save_visualization=False):
        self.calls.append((images, save_visualization))
        return [self.regions], {}

    def stop(self):
        self.stopped = True


async def assert_async_fails(awaitable, contains: str):
    try:
        await awaitable
    except Exception as error:
        assert contains.casefold() in str(error).casefold()
    else:
        raise AssertionError("Expected awaitable to fail")


def make_pdf(path: Path, labels=("page",), password: str | None = None):
    document = pymupdf.open()
    for label in labels:
        page = document.new_page()
        page.insert_text((72, 72), label)
    if password is None:
        document.save(path)
    else:
        document.save(
            path,
            encryption=pymupdf.PDF_ENCRYPT_AES_256,
            owner_pw="owner-password",
            user_pw=password,
        )
    document.close()


def make_image(path: Path, label: str = "image"):
    document = pymupdf.open()
    page = document.new_page(width=240, height=120)
    page.insert_text((24, 60), label)
    page.get_pixmap(alpha=False).save(path)
    document.close()

In [ ]:
# | hide
def test_source_discovery_and_mapping():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        (root / "nested").mkdir()
        (root / ".md").mkdir()
        make_pdf(root / "B.PDF")
        make_pdf(root / "nested" / "a.pdf")
        make_image(root / "nested" / "C.PNG")
        make_image(root / "photo.JpEg")
        make_pdf(root / ".md" / "ignored.pdf")
        make_image(root / ".md" / "ignored.jpg")
        generated_images = [
            root / ".md.v2" / "old.assets" / "ignored.png",
            root / ".md_dashscope" / "ignored.jpeg",
            root / "published.assets" / "ignored.PNG",
            root / ".layout-work" / "ignored.jpg",
        ]
        for generated_image in generated_images:
            generated_image.parent.mkdir(parents=True, exist_ok=True)
            make_image(generated_image)
        (root / "notes.txt").write_text("not a PDF", encoding="utf-8")

        jobs = _ocr_jobs(root)
        test_eq(
            [source.relative_to(root).as_posix() for source, _ in jobs],
            ["B.PDF", "nested/a.pdf", "nested/C.PNG", "photo.JpEg"],
        )
        test_eq(
            [target.relative_to(root).as_posix() for _, target in jobs],
            [".md/B.md", ".md/nested/a.md", ".md/nested/C.md", ".md/photo.md"],
        )


def test_output_collision_is_rejected():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        make_pdf(root / "same.pdf")
        make_image(root / "same.PNG")
        test_fail(lambda: _ocr_jobs(root), contains="output collision")


test_source_discovery_and_mapping()
test_output_collision_is_rejected()

In [ ]:
# | hide
def test_ocr_pdf_writes_ordered_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "two-pages.pdf"
        markdown_path = root / ".md" / "two-pages.md"
        make_pdf(pdf_path, ("first", "second"))
        client = FakeOllamaClient(("# First", "Second"))

        result = ocr_pdf(pdf_path, markdown_path, client=client)

        test_eq(result.status, "processed")
        test_eq(result.pages_total, 2)
        test_eq(result.pages_completed, 2)
        test_eq(
            markdown_path.read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\n# First\n\n<!-- Page 2 -->\n\nSecond\n",
        )
        test_eq(len(client.chat_calls), 2)
        for call in client.chat_calls:
            test_eq(call["model"], "glm-ocr")
            test_eq(call["options"], {"temperature": 0})
            test_eq(call["messages"][0]["content"], "Text Recognition:")
            assert isinstance(call["messages"][0]["images"][0], bytes)


def test_skip_and_overwrite():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "one.pdf"
        markdown_path = root / "one.md"
        make_pdf(pdf_path)
        markdown_path.write_text("existing", encoding="utf-8")

        skipped_client = FakeOllamaClient()
        skipped = ocr_pdf(pdf_path, markdown_path, client=skipped_client)
        test_eq(skipped.status, "skipped")
        test_eq(skipped_client.chat_calls, [])
        test_eq(markdown_path.read_text(encoding="utf-8"), "existing")

        overwritten = ocr_pdf(
            pdf_path,
            markdown_path,
            client=FakeOllamaClient(("replacement",)),
            overwrite=True,
        )
        test_eq(overwritten.status, "processed")
        assert "replacement" in markdown_path.read_text(encoding="utf-8")


test_ocr_pdf_writes_ordered_markdown()
test_skip_and_overwrite()

In [ ]:
# | hide
def test_ocr_image_writes_markdown_for_both_providers():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        png_path = root / "scan.png"
        png_markdown = root / "scan.md"
        make_image(png_path, "local image")
        ollama_client = FakeOllamaClient(("# Local image",))

        local = ocr_image(png_path, png_markdown, client=ollama_client)

        test_eq(local.status, "processed")
        test_eq(local.source_path, png_path.resolve())
        test_eq((local.pages_total, local.pages_completed), (1, 1))
        test_eq(
            png_markdown.read_text(encoding="utf-8"),
            "<!-- Image -->\n\n# Local image\n",
        )
        image_bytes = ollama_client.chat_calls[0]["messages"][0]["images"][0]
        assert image_bytes.startswith(b"\x89PNG\r\n\x1a\n")

        jpg_path = root / "photo.jpg"
        jpg_markdown = root / "photo.md"
        make_image(jpg_path, "cloud image")
        dashscope_client = FakeOpenAIClient(("Cloud image",))
        cloud = ocr_image(
            jpg_path,
            jpg_markdown,
            client=dashscope_client,
            provider="dashscope",
        )

        test_eq(cloud.status, "processed")
        data_url = dashscope_client.calls[0]["messages"][0]["content"][0]["image_url"]["url"]
        assert data_url.startswith("data:image/png;base64,")
        assert base64.b64decode(data_url.split(",", 1)[1]).startswith(b"\x89PNG")


def test_pillow_fallback_decodes_transparent_png():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "keyshot-style.png"
        markdown_path = root / "keyshot-style.md"
        image = Image.new("RGBA", (4, 3), (20, 40, 60, 0))
        image.putpixel((1, 1), (0, 0, 0, 255))
        image.save(image_path)
        client = FakeOllamaClient(("Fallback image",))
        with patch(
            __name__ + "._load_image_pixmap_with_pymupdf",
            side_effect=RuntimeError("simulated MuPDF PNG decoder failure"),
        ):
            result = ocr_image(image_path, markdown_path, client=client)

        assert result.status == "processed", result.error
        sent_png = client.chat_calls[0]["messages"][0]["images"][0]
        decoded = pymupdf.Pixmap(sent_png)
        test_eq((decoded.width, decoded.height, decoded.alpha), (4, 3, 0))
        test_eq(decoded.samples[:3], b"\xff\xff\xff")


def test_pillow_large_image_warning_becomes_a_file_error():
    with TemporaryDirectory() as temporary_directory:
        image_path = Path(temporary_directory) / "large.png"
        Image.new("RGB", (10, 10), "white").save(image_path)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            with patch.object(Image, "MAX_IMAGE_PIXELS", 60):
                test_fail(
                    lambda: _load_image_pixmap_with_pillow(image_path),
                    contains="60-pixel safety limit",
                )
        test_eq(caught, [])


def test_in_memory_layout_crop_suppresses_decompression_warning():
    image = Image.new("RGB", (10, 10), "white")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        with patch.object(Image, "MAX_IMAGE_PIXELS", 60):
            cropped = _crop_pil_image(image, (0, 0, 10, 10))
    test_eq(cropped.size, (10, 10))
    test_eq(caught, [])
    test_fail(
        lambda: _crop_pil_image(image, (0, 0, 11, 10)),
        contains="outside",
    )


async def test_folder_processes_mixed_pdfs_and_images():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "document.pdf")
        make_image(root / "scan.png")
        client = FakeAsyncOllamaClient(("PDF text", "Image text"))

        captured_output = StringIO()
        captured_progress = StringIO()
        with (
            patch(__name__ + "._running_in_notebook", return_value=False),
            redirect_stdout(captured_output),
            redirect_stderr(captured_progress),
        ):
            results = await ocr_folder(root, client=client, max_concurrency=2)

        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq([result.source_path.name for result in results], ["document.pdf", "scan.png"])
        assert (root / ".md" / "document.md").is_file()
        assert (root / ".md" / "scan.md").is_file()
        test_eq(client.show_calls, ["glm-ocr"])
        test_eq(client.max_active_calls, 2)
        timing_output = captured_output.getvalue()
        for source_name in ("document.pdf", "scan.png"):
            source_path = (root / source_name).resolve()
            assert f"OCR task: {source_path} | status=processed | elapsed=" in timing_output
        progress_output = captured_progress.getvalue()
        assert "OCR pages:" in progress_output
        assert "OCR files (ollama" in progress_output
        assert str((root / "scan.png").resolve()) in progress_output


async def test_folder_hides_skipped_files_from_progress():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        skipped_source = root / "already-done.pdf"
        processed_source = root / "new.pdf"
        make_pdf(skipped_source)
        make_pdf(processed_source)
        skipped_target = root / ".md" / "already-done.md"
        skipped_target.parent.mkdir()
        skipped_target.write_text("existing", encoding="utf-8")
        old_mtime_ns = 1_700_000_000_000_000_000
        new_mtime_ns = old_mtime_ns + 1_000_000_000
        os.utime(skipped_source, ns=(old_mtime_ns, old_mtime_ns))
        os.utime(processed_source, ns=(new_mtime_ns, new_mtime_ns))
        client = FakeAsyncOllamaClient(("New text",))

        captured_output = StringIO()
        captured_progress = StringIO()
        with (
            patch(__name__ + "._running_in_notebook", return_value=False),
            redirect_stdout(captured_output),
            redirect_stderr(captured_progress),
        ):
            results = await ocr_folder(root, client=client, max_concurrency=2)

        test_eq([result.status for result in results], ["skipped", "processed"])
        progress_output = captured_progress.getvalue()
        assert str(skipped_source.resolve()) not in progress_output
        assert str(processed_source.resolve()) in progress_output
        output = captured_output.getvalue()
        assert "1 skipped" in output
        report = output.split("OCR files (newest to oldest", 1)[1]
        assert f"OCR file: {processed_source.resolve()} | status=processed" in report
        assert f"OCR file: {skipped_source.resolve()} | status=skipped" in report
        assert report.index(str(processed_source.resolve())) < report.index(
            str(skipped_source.resolve())
        )


async def test_interrupted_folder_reports_completed_and_skipped_files():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        skipped_source = root / "a-skipped.png"
        completed_source = root / "b-completed.png"
        pending_source = root / "c-pending.png"
        for source in (skipped_source, completed_source, pending_source):
            make_image(source)
        skipped_target = root / ".md" / "a-skipped.md"
        skipped_target.parent.mkdir()
        skipped_target.write_text("existing", encoding="utf-8")
        client = FakeAsyncOllamaClient(
            ((0.01, "Completed"), (2.0, "Pending"))
        )
        captured_output = StringIO()

        with redirect_stdout(captured_output), redirect_stderr(StringIO()):
            task = asyncio.create_task(
                ocr_folder(
                    root,
                    client=client,
                    max_concurrency=2,
                    show_page_progress=False,
                )
            )
            completed_target = root / ".md" / "b-completed.md"
            for _ in range(200):
                if completed_target.is_file():
                    break
                await asyncio.sleep(0.01)
            else:
                raise AssertionError("Fast OCR task did not complete")
            task.cancel()
            try:
                await task
            except asyncio.CancelledError:
                pass

        output = captured_output.getvalue()
        assert "OCR interrupted: 1 processed" in output
        assert f"OCR file: {completed_source.resolve()} | status=processed" in output
        assert f"OCR file: {skipped_source.resolve()} | status=skipped" in output
        assert str(pending_source.resolve()) not in output


def test_notebook_progress_renders_page_and_file_rows():
    updates: list[str] = []

    class FakeDisplayHandle:
        def update(self, content):
            updates.append(content.data)

    def fake_display(content, *, display_id):
        test_eq(display_id, True)
        updates.append(content.data)
        return FakeDisplayHandle()

    with (
        patch(__name__ + "._running_in_notebook", return_value=True),
        patch("IPython.display.display", side_effect=fake_display),
    ):
        progress = _OCRFolderProgress(2, "OCR files (test)", True)
        progress.start_file(0, Path("older.pdf"))
        progress.set_pages_total(0, 4)
        progress.start_file(1, Path("newest.pdf"), 2)
        progress.complete_page(1, 1, 2, 0.25)
        live_html = updates[-1]
        progress.finish_file(1)
        progress.finish_file(0)
        progress.close()

    assert "OCR pages: newest.pdf" in live_html
    assert "OCR files (test)" in live_html
    assert live_html.index("OCR pages: newest.pdf") < live_html.index(
        "OCR files (test)"
    )
    assert "1/2" in live_html
    assert "<progress" in live_html


async def test_pdf_pages_run_concurrently_and_keep_markdown_order():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "multipage.pdf"
        make_pdf(pdf_path, labels=("one", "two", "three", "four"))
        responses = (
            (0.04, "First"),
            (0.01, "Second"),
            (0.02, "Third"),
            (0.01, "Fourth"),
        )
        client = FakeAsyncOllamaClient(responses)

        captured_output = StringIO()
        captured_progress = StringIO()
        with (
            patch(__name__ + "._running_in_notebook", return_value=False),
            redirect_stdout(captured_output),
            redirect_stderr(captured_progress),
        ):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=3,
                page_concurrency=3,
            )

        test_eq(results[0].status, "processed")
        test_eq((results[0].pages_total, results[0].pages_completed), (4, 4))
        test_eq(client.max_active_calls, 3)
        test_eq(
            (root / ".md" / "multipage.md").read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\nFirst\n\n"
            "<!-- Page 2 -->\n\nSecond\n\n"
            "<!-- Page 3 -->\n\nThird\n\n"
            "<!-- Page 4 -->\n\nFourth\n",
        )
        progress_output = captured_progress.getvalue()
        assert f"OCR pages: {pdf_path.resolve()}" in progress_output
        assert "4/4" in progress_output
        assert "last=" in progress_output


async def test_request_limit_is_global_across_pdfs():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "a.pdf", labels=("a1", "a2", "a3"))
        make_pdf(root / "b.pdf", labels=("b1", "b2", "b3"))
        client = FakeAsyncOllamaClient(("page",) * 6, delay_s=0.02)

        with redirect_stdout(StringIO()):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=2,
                page_concurrency=3,
                show_page_progress=False,
            )

        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq(len(client.chat_calls), 6)
        test_eq(client.max_active_calls, 2)


async def test_ollama_pool_uses_both_hosts_and_caps_each_host():
    clients = {
        "http://local:11434": FakeAsyncOllamaClient(("local",) * 12, delay_s=0.02),
        "http://remote:11434": FakeAsyncOllamaClient(("remote",) * 12, delay_s=0.02),
    }
    pool = _AsyncOllamaPool(
        ["http://local:11434/", "http://remote:11434"],
        request_timeout_s=180,
        max_concurrency=8,
        client_factory=lambda host, _timeout: clients[host],
    )

    await pool.show("glm-ocr")
    await asyncio.gather(*(pool.chat(model="glm-ocr") for _ in range(12)))

    test_eq(pool.active_hosts, tuple(clients))
    test_eq(pool.request_slots, 8)
    assert all(client.chat_calls for client in clients.values())
    assert all(client.max_active_calls <= 4 for client in clients.values())
    await pool.close()
    assert all(client.closed for client in clients.values())


async def test_ollama_pool_excludes_an_unavailable_host():
    healthy = FakeAsyncOllamaClient(("ok",) * 6, delay_s=0.02)
    unavailable = FakeAsyncOllamaClient(show_error=RuntimeError("offline"))
    clients = {
        "http://healthy:11434": healthy,
        "http://offline:11434": unavailable,
    }
    pool = _AsyncOllamaPool(
        list(clients),
        request_timeout_s=180,
        max_concurrency=8,
        client_factory=lambda host, _timeout: clients[host],
    )

    await pool.show("glm-ocr")
    await asyncio.gather(*(pool.chat(model="glm-ocr") for _ in range(6)))

    test_eq(pool.active_hosts, ("http://healthy:11434",))
    test_eq(pool.request_slots, 4)
    test_eq(pool.unavailable_hosts[0][0], "http://offline:11434")
    test_eq(len(unavailable.chat_calls), 0)
    test_eq(healthy.max_active_calls, 4)
    await pool.close()


test_ocr_image_writes_markdown_for_both_providers()
test_pillow_fallback_decodes_transparent_png()
test_pillow_large_image_warning_becomes_a_file_error()
test_in_memory_layout_crop_suppresses_decompression_warning()
await test_folder_processes_mixed_pdfs_and_images()
await test_folder_hides_skipped_files_from_progress()
await test_interrupted_folder_reports_completed_and_skipped_files()
test_notebook_progress_renders_page_and_file_rows()
await test_pdf_pages_run_concurrently_and_keep_markdown_order()
await test_request_limit_is_global_across_pdfs()
await test_ollama_pool_uses_both_hosts_and_caps_each_host()
await test_ollama_pool_excludes_an_unavailable_host()

In [ ]:
# | hide
def test_structured_layout_writes_markdown_assets_and_sidecar():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "sheet.png"
        markdown_path = root / "sheet.md"
        make_image(image_path, "structured sheet")
        detector = FakeLayoutDetector(
            [
                {"index": 0, "label": "doc_title", "score": 0.99, "bbox_2d": [0, 0, 1000, 250], "task_type": "text"},
                {"index": 1, "label": "table", "score": 0.98, "bbox_2d": [0, 250, 500, 1000], "task_type": "table"},
                {"index": 2, "label": "image", "score": 0.97, "bbox_2d": [500, 250, 1000, 1000], "task_type": "figure"},
            ]
        )
        client = FakeOllamaClient(
            ("SCARA Robot", "<table><tr><td>4 kg</td></tr></table>")
        )

        result = ocr_image(
            image_path,
            markdown_path,
            client=client,
            layout_mode="pp-doclayout",
            layout_detector=detector,
            embed_page_image=True,
        )

        assert result.status == "processed", result.error
        test_eq(
            [call["messages"][0]["content"] for call in client.chat_calls],
            ["Text Recognition:", "Table Recognition:"],
        )
        test_eq(
            client.chat_calls[0]["options"],
            {"temperature": 0, "num_predict": 1024, "repeat_penalty": 1.1},
        )
        test_eq(
            client.chat_calls[1]["options"],
            {"temperature": 0, "num_predict": 4096, "repeat_penalty": 1.1},
        )
        markdown = markdown_path.read_text(encoding="utf-8")
        assert "# SCARA Robot" in markdown
        assert "<table><tr><td>4 kg</td></tr></table>" in markdown
        assert "![Table 2](<sheet.assets/page-0001-region-002-table.png>)" in markdown
        assert "![Image 3](<sheet.assets/page-0001-region-003-image.png>)" in markdown
        assert "![Original page 1](<sheet.assets/page-0001.png>)" in markdown
        assert "layout-region page=1 index=2 label=table" in markdown
        assets = sorted((root / "sheet.assets").glob("*.png"))
        test_eq(len(assets), 3)
        layout = json.loads((root / "sheet.layout.json").read_text(encoding="utf-8"))
        test_eq(layout["schema_version"], 2)
        test_eq(layout["settings"]["response_validation_version"], 2)
        test_eq(layout["layout_model"], "PaddlePaddle/PP-DocLayoutV3_safetensors")
        test_eq(layout["pages"][0]["regions"][0]["bbox"], [0, 0, 240, 30])
        test_eq(
            [region["task_type"] for region in layout["pages"][0]["regions"]],
            ["text", "table", "figure"],
        )
        test_eq(
            [region["status"] for region in layout["pages"][0]["regions"]],
            ["completed", "completed", "preserved"],
        )
        test_eq(
            [region.get("max_output_tokens") for region in layout["pages"][0]["regions"]],
            [1024, 4096, None],
        )
        assert detector.calls[0][0][0].size == (240, 120)
        assert not detector.stopped


def test_repeated_and_truncated_layout_responses_become_partial():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "degenerate.png"
        target = root / "degenerate.md"
        make_image(image_path)
        detector = FakeLayoutDetector(
            [
                {"index": 0, "label": "text", "score": 0.9, "bbox_2d": [0, 0, 500, 1000], "task_type": "text"},
                {"index": 1, "label": "paragraph_title", "score": 0.9, "bbox_2d": [500, 0, 1000, 1000], "task_type": "text"},
            ]
        )
        repeated = "SCARA robot\n" + "```\n" * 20
        truncated = SimpleNamespace(
            message=SimpleNamespace(content="Useful but incomplete"),
            done_reason="length",
        )
        result = ocr_image(
            image_path,
            target,
            client=FakeOllamaClient((repeated, truncated)),
            layout_mode="pp-doclayout",
            layout_detector=detector,
        )

        test_eq(result.status, "partial")
        test_eq((result.regions_total, result.regions_completed, result.regions_failed), (2, 1, 1))
        markdown = target.read_text(encoding="utf-8")
        test_eq(markdown.count("SCARA robot"), 1)
        assert "pathological repeated response" not in markdown
        assert "output-token limit" in markdown
        assert repeated not in markdown
        assert "Useful but incomplete" not in markdown
        assert "![Failed paragraph_title region 2]" in markdown
        assert "\n## \n" not in markdown
        assets = sorted((root / "degenerate.assets").glob("*.png"))
        test_eq(len(assets), 2)
        layout = json.loads((root / "degenerate.layout.json").read_text(encoding="utf-8"))
        test_eq(
            [region["status"] for region in layout["pages"][0]["regions"]],
            ["recovered", "failed"],
        )
        test_eq(layout["pages"][0]["regions"][0]["content"], "SCARA robot")
        assert layout["pages"][0]["regions"][0]["recovery"]
        test_eq(layout["pages"][0]["regions"][1]["content"], "")

        table_content = "<table><tr><td>-</td></tr></table>"
        table_response = SimpleNamespace(
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content=table_content),
                    finish_reason="stop",
                )
            ]
        )
        test_eq(
            _validated_layout_content(
                table_response, "dashscope", table_content, "table"
            ),
            (table_content, None),
        )
        repeated_table = table_content * 20
        truncated_table_response = SimpleNamespace(
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content=repeated_table),
                    finish_reason="length",
                )
            ]
        )
        recovered_table, table_recovery = _validated_layout_content(
            truncated_table_response,
            "dashscope",
            repeated_table,
            "table",
        )
        test_eq(recovered_table, table_content)
        assert table_recovery
        empty_footer, footer_recovery = _validated_layout_content(
            SimpleNamespace(
                message=SimpleNamespace(content="---\n" * 20),
                done_reason="length",
            ),
            "ollama",
            "---\n" * 20,
            "text",
            "footer",
        )
        test_eq(empty_footer, "")
        assert footer_recovery
        continuous_loop = "maximum payload 4 kg, reach 400 mm;" * 20
        assert "32-character block" in (
            _layout_repetition_reason(continuous_loop) or ""
        )
        test_eq(
            _layout_repetition_reason(
                "A normal OCR paragraph with no repeated sequence."
            ),
            None,
        )


async def test_async_repeated_layout_response_is_recovered():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "async-loop.png"
        target = root / "async-loop.md"
        make_image(image_path)
        detector = FakeLayoutDetector(
            [{"index": 0, "label": "text", "score": 0.9, "bbox_2d": [0, 0, 1000, 1000], "task_type": "text"}]
        )
        client = FakeAsyncOllamaClient(("Repeated sentence.\n" * 20,))
        result = await _ocr_image_async(
            image_path,
            target,
            client=client,
            layout_mode="pp-doclayout",
            layout_detector=detector,
        )

        test_eq(result.status, "processed")
        test_eq((result.regions_completed, result.regions_failed), (1, 0))
        test_eq(client.chat_calls[0]["options"]["num_predict"], 1024)
        test_eq(target.read_text(encoding="utf-8").count("Repeated sentence."), 1)
        assert len(list((root / "async-loop.assets").glob("*.png"))) == 1
        layout = json.loads((root / "async-loop.layout.json").read_text(encoding="utf-8"))
        test_eq(layout["pages"][0]["regions"][0]["status"], "recovered")


def test_structured_dashscope_payload_and_failed_bundle_cleanup():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "cloud.png"
        make_image(image_path)
        detector = FakeLayoutDetector(
            [{"index": 0, "label": "table", "score": 0.9, "bbox_2d": [0, 0, 1000, 1000], "task_type": "table"}]
        )
        cloud_client = FakeOpenAIClient(("<table></table>",))
        cloud_target = root / "cloud.md"
        result = ocr_image(
            image_path,
            cloud_target,
            client=cloud_client,
            provider="dashscope",
            layout_mode="pp-doclayout",
            layout_detector=detector,
        )
        assert result.status == "processed", result.error
        call = cloud_client.calls[0]
        test_eq(call["extra_body"], {"enable_thinking": False, "vl_high_resolution_images": True})
        test_eq(call["max_tokens"], 4096)
        test_eq(call["messages"][0]["content"][1]["text"], "qwenvl markdown")

        failed_target = root / "failed.md"
        failed_detector = FakeLayoutDetector(
            [
                {"index": 0, "label": "text", "score": 0.9, "bbox_2d": [0, 0, 500, 1000], "task_type": "text"},
                {"index": 1, "label": "table", "score": 0.9, "bbox_2d": [500, 0, 1000, 1000], "task_type": "table"},
            ]
        )
        failed = ocr_image(
            image_path,
            failed_target,
            client=FakeOllamaClient(("text", RuntimeError("region failed"))),
            layout_mode="pp-doclayout",
            layout_detector=failed_detector,
        )
        test_eq(failed.status, "partial")
        test_eq((failed.regions_total, failed.regions_completed, failed.regions_failed), (2, 1, 1))
        failed_markdown = failed_target.read_text(encoding="utf-8")
        assert "OCR failed for" in failed_markdown
        assert "![Table 2](<failed.assets/page-0001-region-002-table.png>)" in failed_markdown
        failed_layout = json.loads((root / "failed.layout.json").read_text(encoding="utf-8"))
        test_eq(failed_layout["status"], "partial")
        test_eq(failed_layout["pages"][0]["regions"][1]["status"], "failed")
        assert (root / "failed.assets").is_dir()
        assert not (root / "failed.partial.md").exists()
        assert not (root / ".failed.layout-work").exists()


async def test_folder_reuses_injected_layout_detector():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_image(root / "a.png")
        make_image(root / "b.png")
        detector = FakeLayoutDetector(
            [{"index": 0, "label": "text", "score": 0.9, "bbox_2d": [0, 0, 1000, 1000], "task_type": "text"}]
        )
        client = FakeAsyncOllamaClient(("A", "B"))
        with redirect_stdout(StringIO()):
            results = await ocr_folder(
                root,
                client=client,
                layout_mode="pp-doclayout",
                layout_detector=detector,
                max_concurrency=2,
                show_page_progress=False,
            )
        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq(len(detector.calls), 2)
        test_eq(
            [call["messages"][0]["content"] for call in client.chat_calls],
            ["Text Recognition:", "Text Recognition:"],
        )
        assert (root / ".md" / "a.layout.json").is_file()
        assert (root / ".md" / "b.layout.json").is_file()
        assert not detector.stopped


async def test_partial_checkpoint_survives_and_resumes_regions():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "resume.png"
        target = root / "resume.md"
        make_image(image_path)
        detector = FakeLayoutDetector(
            [
                {"index": 0, "label": "text", "score": 0.9, "bbox_2d": [0, 0, 500, 1000], "task_type": "text"},
                {"index": 1, "label": "text", "score": 0.9, "bbox_2d": [500, 0, 1000, 1000], "task_type": "text"},
            ]
        )
        first_client = FakeAsyncOllamaClient(((0.01, "First"), (2.0, "Second")))
        task = asyncio.create_task(
            _ocr_image_async(
                image_path,
                target,
                client=first_client,
                layout_mode="pp-doclayout",
                layout_detector=detector,
            )
        )
        partial_path = root / "resume.partial.md"
        checkpoint_path = root / ".resume.layout-work" / "checkpoint.json"
        for _ in range(200):
            if checkpoint_path.is_file():
                checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
                if _layout_counts(checkpoint)[1] == 1:
                    break
            await asyncio.sleep(0.01)
        else:
            raise AssertionError("First region was not checkpointed")
        assert partial_path.is_file()
        assert "OCR processing is incomplete" in partial_path.read_text(encoding="utf-8")
        task.cancel()
        try:
            await task
        except asyncio.CancelledError:
            pass

        resumed_client = FakeAsyncOllamaClient(("Second",))
        resumed = await _ocr_image_async(
            image_path,
            target,
            client=resumed_client,
            layout_mode="pp-doclayout",
            layout_detector=detector,
        )
        test_eq(resumed.status, "processed")
        test_eq(len(resumed_client.chat_calls), 1)
        final_markdown = target.read_text(encoding="utf-8")
        assert "First" in final_markdown and "Second" in final_markdown
        assert not partial_path.exists()
        assert not checkpoint_path.parent.exists()


def test_incompatible_checkpoint_requires_overwrite():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "settings.png"
        target = root / "settings.md"
        make_image(source)
        settings = _layout_settings(
            provider="ollama",
            model="glm-ocr",
            prompt=None,
            dpi=None,
            layout_model=_DEFAULT_LAYOUT_MODEL,
            layout_threshold=0.3,
            embed_page_image=False,
            max_output_tokens=4096,
        )
        _, _, original = _prepare_layout_workspace(
            source, target, pages_total=1, settings=settings,
            resume_partial=True, overwrite=False,
        )
        changed_settings = {**settings, "max_output_tokens": 2048}
        test_fail(
            lambda: _prepare_layout_workspace(
                source, target, pages_total=1, settings=changed_settings,
                resume_partial=True, overwrite=False,
            ),
            contains="incompatible",
        )
        _, _, replaced = _prepare_layout_workspace(
            source, target, pages_total=1, settings=changed_settings,
            resume_partial=True, overwrite=True,
        )
        test_eq(original["settings"]["max_output_tokens"], 4096)
        test_eq(replaced["settings"]["max_output_tokens"], 2048)


def test_layout_publication_rolls_back_on_failure():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        stage = root / "stage"
        stage.mkdir()
        target = root / "document.md"
        sidecar = root / "document.layout.json"
        assets = root / "document.assets"
        target.write_text("old markdown", encoding="utf-8")
        sidecar.write_text("old sidecar", encoding="utf-8")
        assets.mkdir()
        (assets / "old.png").write_bytes(b"old")
        staged_assets = stage / "document.assets"
        staged_assets.mkdir()
        (staged_assets / "new.png").write_bytes(b"new")
        original_replace = Path.replace

        def fail_on_sidecar(path, destination):
            if path == stage / "document.layout.json":
                raise OSError("simulated publication failure")
            return original_replace(path, destination)

        with patch.object(Path, "replace", fail_on_sidecar):
            test_fail(
                lambda: _publish_layout_bundle(
                    stage, target, "new markdown", {"schema_version": 2}
                ),
                contains="simulated publication failure",
            )
        test_eq(target.read_text(encoding="utf-8"), "old markdown")
        test_eq(sidecar.read_text(encoding="utf-8"), "old sidecar")
        assert (assets / "old.png").is_file()
        assert not (assets / "new.png").exists()


async def test_folder_output_lock_rejects_a_second_batch():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_image(root / "locked.png")
        lock_path = root / ".ocr-folder.lock"
        with _exclusive_output_lock(lock_path, "test batch"):
            await assert_async_fails(
                ocr_folder(root, client=FakeAsyncOllamaClient(("unused",))),
                contains="another OCR process",
            )


test_structured_layout_writes_markdown_assets_and_sidecar()
test_repeated_and_truncated_layout_responses_become_partial()
await test_async_repeated_layout_response_is_recovered()
test_structured_dashscope_payload_and_failed_bundle_cleanup()
await test_folder_reuses_injected_layout_detector()
await test_partial_checkpoint_survives_and_resumes_regions()
test_incompatible_checkpoint_requires_overwrite()
test_layout_publication_rolls_back_on_failure()
await test_folder_output_lock_rejects_a_second_batch()

In [ ]:
# | hide
def test_provider_and_model_resolution():
    assert (PROJ_ROOT / "pyproject.toml").is_file()
    test_eq(_resolve_provider("OLLAMA"), "ollama")
    test_fail(lambda: _resolve_provider("unknown"), contains="provider must")
    test_eq(_resolve_layout_mode("PP-DOCLAYOUT"), "pp-doclayout")
    test_fail(lambda: _resolve_layout_mode("unknown"), contains="layout_mode")
    test_eq(_resolve_model("ollama", None), "glm-ocr")
    test_fail(lambda: _resolve_model("ollama", "  "), contains="model must")
    test_eq(_resolve_prompt("dashscope", " custom "), "custom")
    test_eq(
        _normalize_ollama_hosts(["http://local:11434/", "http://local:11434", "http://remote:11434"]),
        ("http://local:11434", "http://remote:11434"),
    )
    test_fail(lambda: _normalize_ollama_hosts([]), contains="At least one")
    test_fail(lambda: _normalize_ollama_hosts([" "]), contains="non-empty")

    with patch.dict(os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False):
        test_eq(_resolve_model("dashscope", None), "qwen3.7-plus")
    with patch.dict(
        os.environ,
        {"OPENAILIKED_OCR_MODEL": "environment-model"},
        clear=False,
    ):
        test_eq(_resolve_model("dashscope", None), "environment-model")
        test_eq(_resolve_model("dashscope", "explicit-model"), "explicit-model")


test_provider_and_model_resolution()

In [ ]:
# | hide
def test_dashscope_ocr_payload_and_response():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "cloud.pdf"
        markdown_path = root / "cloud.md"
        make_pdf(pdf_path, ("cloud OCR",))
        client = FakeOpenAIClient(("# Cloud OCR",))
        with patch.dict(
            os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False
        ):
            result = ocr_pdf(
                pdf_path,
                markdown_path,
                client=client,
                provider="dashscope",
            )

        test_eq(result.status, "processed")
        test_eq(len(client.calls), 1)
        call = client.calls[0]
        test_eq(call["model"], "qwen3.7-plus")
        test_eq(call["temperature"], 0)
        test_eq(call["extra_body"], {"enable_thinking": False})
        content = call["messages"][0]["content"]
        image_url = content[0]["image_url"]["url"]
        assert image_url.startswith("data:image/png;base64,")
        assert base64.b64decode(image_url.split(",", 1)[1]).startswith(
            b"\x89PNG\r\n\x1a\n"
        )
        assert "Markdown" in content[1]["text"]
        assert "# Cloud OCR" in markdown_path.read_text(encoding="utf-8")


test_dashscope_ocr_payload_and_response()

In [ ]:
# | hide
class CompressiblePixmap:
    def tobytes(self, output="png", jpg_quality=95):
        return b"x" * (32 if output == "png" else 4)


class OversizedPixmap:
    def tobytes(self, output="png", jpg_quality=95):
        return b"x" * 32


def test_dashscope_image_size_fallback_and_failure():
    with patch.dict(globals(), {"_DASHSCOPE_MAX_DATA_URL_BYTES": 60}):
        data_url = _dashscope_image_data_url(CompressiblePixmap())
        assert data_url.startswith("data:image/jpeg;base64,")
        test_fail(
            lambda: _dashscope_image_data_url(OversizedPixmap()),
            contains="10 MiB",
        )


test_dashscope_image_size_fallback_and_failure()

In [ ]:
# | hide
def test_failures_do_not_publish_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "empty-response.pdf"
        markdown_path = root / "empty-response.md"
        make_pdf(pdf_path)
        result = ocr_pdf(pdf_path, markdown_path, client=FakeOllamaClient(("  ",)))
        test_eq(result.status, "failed")
        assert "empty response" in (result.error or "")
        assert not markdown_path.exists()
        assert not list(root.glob("*.tmp"))


def test_corrupt_image_fails_without_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        image_path = root / "corrupt.png"
        markdown_path = root / "corrupt.md"
        image_path.write_bytes(b"not an image")

        result = ocr_image(image_path, markdown_path, client=FakeOllamaClient())

        test_eq(result.status, "failed")
        assert "Could not decode image" in (result.error or "")
        assert not markdown_path.exists()


def test_corrupt_and_encrypted_pdfs_fail_cleanly():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        corrupt_pdf = root / "corrupt.pdf"
        corrupt_pdf.write_bytes(b"not a PDF")
        encrypted_pdf = root / "encrypted.pdf"
        make_pdf(encrypted_pdf, password="secret")

        corrupt = ocr_pdf(corrupt_pdf, root / "corrupt.md", client=FakeOllamaClient())
        encrypted = ocr_pdf(encrypted_pdf, root / "encrypted.md", client=FakeOllamaClient())
        test_eq(corrupt.status, "failed")
        test_eq(encrypted.status, "failed")
        assert "password" in (encrypted.error or "").lower()
        assert not (root / "corrupt.md").exists()
        assert not (root / "encrypted.md").exists()


async def test_folder_continues_after_a_failed_pdf():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "a.pdf")
        make_pdf(root / "b.pdf")
        client = FakeAsyncOllamaClient((RuntimeError("model failure"), "# B"))

        results = await ocr_folder(root, client=client, max_concurrency=2)

        test_eq([result.status for result in results], ["failed", "processed"])
        assert not (root / ".md" / "a.md").exists()
        assert (root / ".md" / "b.md").exists()
        test_eq(client.show_calls, ["glm-ocr"])


async def test_concurrent_page_failure_does_not_publish_markdown():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_pdf(root / "multipage.pdf", labels=("one", "two", "three"))
        client = FakeAsyncOllamaClient(
            (RuntimeError("page failure"), "Second", "Third")
        )

        with redirect_stdout(StringIO()):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=3,
                page_concurrency=3,
                show_page_progress=False,
            )

        test_eq(results[0].status, "failed")
        assert "page failure" in (results[0].error or "")
        test_eq(client.max_active_calls, 3)
        assert not (root / ".md" / "multipage.md").exists()


async def test_dashscope_failures_are_isolated():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        malformed_pdf = root / "malformed.pdf"
        make_pdf(malformed_pdf)
        malformed_target = root / "malformed.md"
        malformed = ocr_pdf(
            malformed_pdf,
            malformed_target,
            client=FakeOpenAIClient((SimpleNamespace(choices=[]),)),
            provider="dashscope",
        )
        test_eq(malformed.status, "failed")
        assert "empty response" in (malformed.error or "")
        assert not malformed_target.exists()

        make_pdf(root / "a.pdf")
        make_pdf(root / "b.pdf")
        client = FakeAsyncOpenAIClient((RuntimeError("cloud failure"), "# B"))
        with patch.dict(
            os.environ, {"OPENAILIKED_OCR_MODEL": ""}, clear=False
        ):
            results = await ocr_folder(
                root, provider="dashscope", client=client, max_concurrency=2
            )
        test_eq(client.max_active_calls, 2)
        test_eq(client.calls[0]["extra_body"], {"enable_thinking": False})
        result_by_name = {result.pdf_path.name: result for result in results}
        test_eq(result_by_name["a.pdf"].status, "failed")
        test_eq(result_by_name["b.pdf"].status, "processed")
        assert not (root / ".md" / "a.md").exists()
        assert (root / ".md" / "b.md").exists()


test_failures_do_not_publish_markdown()
test_corrupt_image_fails_without_markdown()
test_corrupt_and_encrypted_pdfs_fail_cleanly()
await test_folder_continues_after_a_failed_pdf()
await test_concurrent_page_failure_does_not_publish_markdown()
await test_dashscope_failures_are_isolated()

In [ ]:
# | hide
async def test_folder_validation_and_preflight():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        missing = root / "missing"
        await assert_async_fails(ocr_folder(missing), contains="does not exist")

        empty_client = FakeAsyncOllamaClient()
        test_eq(await ocr_folder(root, client=empty_client), [])
        test_eq(empty_client.show_calls, [])

        make_pdf(root / "document.pdf")
        unavailable_client = FakeAsyncOllamaClient(
            show_error=RuntimeError("offline")
        )
        await assert_async_fails(
            ocr_folder(root, client=unavailable_client),
            contains="ollama pull glm-ocr",
        )

        skipped_target = root / ".md" / "document.md"
        skipped_target.parent.mkdir()
        skipped_target.write_text("existing", encoding="utf-8")
        skipped_client = FakeAsyncOllamaClient()
        skipped_output = StringIO()
        with redirect_stdout(skipped_output), redirect_stderr(StringIO()):
            skipped_results = await ocr_folder(root, client=skipped_client)
        test_eq([result.status for result in skipped_results], ["skipped"])
        test_eq(skipped_client.show_calls, [])
        assert (
            f"OCR file: {(root / 'document.pdf').resolve()} | status=skipped"
            in skipped_output.getvalue()
        )
        skipped_target.unlink()

        await assert_async_fails(
            ocr_folder(root, provider="invalid"), contains="provider must"
        )
        await assert_async_fails(
            ocr_folder(root, max_concurrency=0), contains="max_concurrency"
        )
        await assert_async_fails(
            ocr_folder(root, page_concurrency=0), contains="page_concurrency"
        )
        await assert_async_fails(
            ocr_folder(root, layout_mode="invalid"), contains="layout_mode"
        )
        await assert_async_fails(
            ocr_folder(root, layout_threshold=1.1), contains="layout_threshold"
        )

        with patch.dict(os.environ, {"DASHSCOPE_API_KEY": ""}, clear=False):
            await assert_async_fails(
                ocr_folder(root, provider="dashscope"),
                contains="DASHSCOPE_API_KEY",
            )
        with patch.dict(
            os.environ,
            {
                "DASHSCOPE_API_KEY": "test-key",
                "DASHSCOPE_API_URL": "https://example.invalid/v1",
            },
            clear=False,
        ):
            await assert_async_fails(
                ocr_folder(root, provider="dashscope"),
                contains="OpenAI-compatible",
            )


await test_folder_validation_and_preflight()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

## Restoring tables, figures, lists, and page structure

The current one-request-per-page implementation is optimized for text transcription, not lossless document reconstruction. A representative comparison between [the original SA4A product sheet](<../res/PDF-20260721/01宣传资料/产品样本/单页样本/SA4A 产品单页.jpg>) and [its generated Markdown](<../res/PDF-20260721/.md/01宣传资料/产品样本/单页样本/SA4A 产品单页.md>) shows that the text is mostly present but the document objects and their spatial relationships have been discarded.

| Element | Original | Current Markdown |
|---|---|---|
| Page layout | Two-column product sheet | Flattened text stream |
| Product features | Three-item list | Plain paragraphs |
| Specifications | Grouped three-column table with merged cells | Space-separated lines |
| Figures | Product image and three major engineering-drawing regions | Headings only; no image assets |
| Reading structure | Headings, panels, captions, and footer | Mostly unformatted text |

### Why the structure is lost

1. The Ollama path sends the complete page using only `Text Recognition:`. GLM-OCR exposes separate `Text Recognition:`, `Table Recognition:`, and `Figure Recognition:` modes, so the current request does not perform region-specific recognition. See the [official Ollama GLM-OCR usage](https://ollama.com/library/glm-ocr).
2. The notebook calls the raw model directly and does not run a layout detector. The official GLM-OCR pipeline combines PP-DocLayout-V3 layout analysis, parallel region recognition, and a result formatter that returns Markdown plus JSON layout details. See the [official GLM-OCR SDK](https://github.com/zai-org/GLM-OCR).
3. A chat response can return text or descriptions but cannot return the original figure pixels as new files. Figure regions must be cropped from the source page, saved as assets, and referenced from the Markdown.
4. CommonMark has no representation for columns, absolute coordinates, or merged table cells. Semantic reading order should be represented linearly; exact merged tables require inline HTML, and exact page appearance requires HTML/CSS or an embedded image of the original page.
5. The SA4A image is 5031×3437, approximately 17.29 megapixels, and contains very small engineering annotations. Whole-page visual encoding can downscale those details before recognition. Region crops preserve substantially more effective resolution.

### Recommended reconstruction pipeline

A reliable structured mode should use the original page as the source of truth and retain an intermediate layout representation:

1. Detect page regions and record each region's type, bounding box, page number, confidence, and reading order in a `.layout.json` sidecar.
2. Crop each detected region from the lossless rendered page.
3. Route text, headings, and lists through text recognition and tables/formulas through their specialized prompts. Preserve diagrams as original crops without asking the text model to regenerate or describe their pixels.
4. Save every non-text region as an image asset even when the model also returns a textual description. Never regenerate engineering drawings from model output.
5. Normalize recognized blocks into semantic objects and assemble them in deterministic reading order.
6. Render simple tables as Markdown and tables containing merged cells as inline HTML `<table>` elements.
7. Optionally embed the complete original page at the top as a visual-fidelity fallback.

For example, a source should produce a sibling asset directory:

```text
SA4A 产品单页.md
SA4A 产品单页.layout.json
SA4A 产品单页.assets/
├── product.png
├── movement-range.png
├── installation-top.png
└── installation-bottom.png
```

The resulting document can then preserve semantic structure and original figure pixels:

```markdown
# SCARA机器人

## SA4A系列

最大可搬运质量 4 kg，可达最大半径 400 mm。

## 产品特点

- **性能卓越：** 全新的自主核心部件及算法……
- **扩展性强：** 本体标配多路气管、信号线及 CAT5E 网口……
- **使用便捷：** 轻量、紧凑化的结构设计……

## 规格参数

<table>
  <tr><th>型号</th><th colspan="2">SA4A-4/0.40</th></tr>
  <tr><th rowspan="3">机械臂长（mm）</th><th>J1+J2</th><td>400</td></tr>
  <tr><th>J1</th><td>225</td></tr>
  <tr><th>J2</th><td>175</td></tr>
</table>

## 运动范围

![运动范围](<SA4A 产品单页.assets/movement-range.png>)

## 安装尺寸

![安装尺寸](<SA4A 产品单页.assets/installation-top.png>)

![本体安装尺寸](<SA4A 产品单页.assets/installation-bottom.png>)
```

### Provider-specific changes

#### Local Ollama and GLM-OCR

Prompt changes alone are insufficient because a full page must first be segmented. This notebook retains the Ollama service while running PP-DocLayout locally, sends only text/table/formula crops with their specialized prompts, and follows the official GLM-OCR SDK by marking chart/image regions as `skip` recognition. Those regions are saved losslessly and embedded directly. This avoids the observed `Figure Recognition:` failure mode that produced thousands of empty Markdown fences.

#### DashScope and Qwen

Replace the generic transcription prompt with the documented `qwenvl markdown` or `qwenvl html` document-parsing prompt. Qwen document parsing can preserve table/image position information. See [DashScope document parsing](https://www.alibabacloud.com/help/en/model-studio/vision#document-parsing).

The OpenAI-compatible request should also enable high-resolution visual processing:

```python
extra_body={
    "enable_thinking": False,
    "vl_high_resolution_images": True,
}
```

Without this flag, Qwen3.7's default visual limit is approximately 2.62 megapixels; high-resolution mode raises the limit to 16,777,216 pixels. The source page is slightly larger than that maximum and will still be downscaled, so layout-based cropping remains necessary for small labels and dimension annotations. See [DashScope high-resolution processing](https://www.alibabacloud.com/help/en/model-studio/vision#process-high-resolution-images).

### Implemented notebook extension

The notebook now implements this design through `layout_mode="pp-doclayout"` while preserving `layout_mode="plain"` as the fast path. It loads the official GLM-OCR `PPDocLayoutDetector` once per folder run, uses detector reading order, recognizes text/table/formula crops, embeds figure and table source crops, and publishes a UTF-8 schema-v2 `.layout.json` sidecar with page dimensions, labels, confidence scores, pixel bounding boxes, prompts, assets, content, status, recovery, and errors. Every region is represented in Markdown by a stable page/index/label/bounding-box marker. Individual request failures produce visible warnings and a best-effort final bundle instead of discarding the document. `.partial.md` checkpoints are refreshed after detection and every region, compatible work resumes automatically, and an advisory output-tree lock prevents duplicate updated batches. Structured text responses are capped at 1,024 tokens and checked for provider-reported truncation, excessive Markdown fences, repeated lines, and repeated 32-character blocks. Degenerate text keeps its first stable semantic unit; length-limited table output keeps its first complete `<table>…</table>`. Such regions are marked `recovered`, their crops and audit reasons are preserved in the sidecar, and no visible warning is emitted. Only output that cannot be recovered is omitted and published as a failed region in a `partial` document. The validator version is included in checkpoint settings so older checkpoints are not silently resumed. DashScope region requests continue to use `qwenvl markdown` with `vl_high_resolution_images=True`. Set `embed_page_image=True` when an exact full-page fallback is required. The optional detector runtime is installed with `uv sync --extra ocr-layout`.